Canonical final R3 analysis source; the distributable notebook is generated by make_notebook.py.

Markdown cell 0
# Final R3 reproducible analysis (2026-09-21)

This version formally integrates the reviewer-requested reference-case PSA ICER ranges into
main Table 2, the machine-readable CSV outputs, and `analysis_summary.json`. ICER percentiles
are calculated among draws with positive incremental QALYs, with the probability of non-positive
incremental QALYs reported separately.

# IDJ major-revision analysis (2026-08-14)

This version implements the current International Dental Journal reviewer requests. It is a
**proof-of-concept simulation**, not a decision-grade economic evaluation. The principal new
analyses are: (1) linear, logistic, probit and Emax PPD-to-pocket-closure structural mappings;
(2) same-draw probabilistic sensitivity analysis under every mapping; (3) multistart/Jacobian
identifiability diagnostics for the sparse GCF calibration; (4) an MMP-8-excluded base case;
and (5) rounded, provenance-labelled economic outputs. No independent patient-level PK/GCF
validation dataset is available, so independent validation remains explicitly not performed.

---

# Evidence-audited correction (2026-07-22)

This notebook is a corrected, reproducible revision of the supplied pipeline. It separates four classes of inputs: **reported measurements**, **published estimates**, **calibrated nuisance parameters**, and **structural/illustrative assumptions**. Only the first two are described as evidence.

Major corrections:

1. The tetracycline descriptor is now **octanol/buffer logD at pH 6.5** (doxycycline -0.08; minocycline 0.20), not an incorrectly labelled logP of 0.60/1.10 (Yamauchi et al., 2020; DOI 10.5599/admet.797).
2. The ARESTIN antibacterial threshold is **1 ug/mL**, as defined in the FDA clinical-pharmacology review; the previous 8 ug/mL value was unsupported.
3. ATRIDOX, PerioChip and ARESTIN GCF inputs use reported summary anchors from Stoller et al./FDA, Soskolne et al./FDA, and the ARESTIN label. Where only a plateau summary is public, it is explicitly labelled a summary representation rather than digitized patient-level data.
4. Atridox and Arestin product-specific medium-term WMDs are attributed to the 2025 product-stratified meta-analysis (Soysa et al.; DOI 10.3389/fdmed.2025.1658720), not to the pivotal trials. PerioChip uses Ma & Diao 2020 (DOI 10.1186/s12903-020-01247-8); its 6-month CAL value is corrected from 0.54 to 0.68 mm.
5. The 0.50-0.58 mm estimates from Annisa et al. 2023 compare chlorhexidine chips with **other antimicrobials** and are not a confidence interval for chip+SRP vs SRP. They are removed. The evidence scenario uses the FDA pivotal incremental effect (~0.30 mm at 9 months) and the Ma-Diao meta-analytic estimate (0.75 mm at 6 months), with the time-point/design mismatch stated.
6. The former human-scale claim for the "systemic compartment" was not supportable. It is now an **absorbed sink**; its output is a model-dependent absorbed fraction, not a validated serum concentration or a label-derived "5% of dose".
7. Dose/formulation fits are renamed **calibration to published summary anchors**, not external validation. Known PerioChip and Arestin doses are fixed; the unknown ATRIDOX per-pocket mass is a bounded nuisance scale and is not interpreted.
8. PPD-to-closure, recurrence, costs, utilities, subgroup prevalences/modifiers and health-economic outputs remain structural demonstrations. They are labelled illustrative and must not be presented as clinical or payer estimates without external calibration.

Verified source URLs and identifiers are listed in the final notebook cell and exported in the parameter-provenance table.

---

Periodontal mini-PBPK -> PK/PD -> precision health-economics pipeline (evidence-audited v4)
¶
A single computational chain:
local release -> pocket exposure -> pocket closure -> recurrence -> QALYs & cost (Markov) -> optimal strategy -> value of information (EVPI/EVPPI/EVIC). Machine learning is used only as an internal implementation check.
v3 - reviewer-response revisions (this version)
¶
All changes are switch-driven from the
CFG
block, so every choice is explicit and reversible.
#2 (double counting) - fixed.
Doxycycline host modulation previously entered
both
as a subgroup closure-rescue (raising QALYs through the Markov)
and
as a separate additive oral-QALY. v3 enforces a
single pathway
(
CFG["HM_PATHWAY"]
=
"closure"
|
"utility"
|
"none"
); the two channels are mutually exclusive, so no effect is counted twice.
#1 (host-modulation extrapolation) - demoted to a scenario.
Host-modulation
strength
is a scenario multiplier (
CFG["HM_SCENARIO"]
=
none
/
weaker
/
current
, from the sub-antimicrobial-dose Caton 2000 anchor). A scenario sweep prints the population-optimal strategy under each pathway x strength, making the fragility of the doxycycline result explicit.
#6 (PPD -> closure mapping) - documented + propagated.
The full mapping
closure = clip(P0_SRP + KAPPA_CLOSE*PPD_WMD, floor, PCLOSE_CEIL)
and the Hill response-gradient are documented with sources in
CFG
. The structural parameters
P0_SRP
,
KAPPA_CLOSE
,
PCLOSE_CEIL
are now
sampled in the PSA
and carried through to a dedicated EVPPI group; the Hill gradient
gamma
is exposed for a one-way check.
#10 (analysis unit) - stated + plausibility-checked.
The Markov trace is one representative treated periodontal site (index tooth); the base-CEA step prints the analysis-unit statement and an external plausibility check of each incremental QALY in healthy-day equivalents.
#3 (dominance labelling) - classified in code.
The base-CEA step labels every off-frontier adjunct as
strongly (simple)
vs
extended
dominated.
#8 (resistance-penalty direction) - corrected.
The stewardship narrative now states the correct direction: the penalty falls only on the antibiotic arms, so it
erodes
doxycycline's lead; chlorhexidine overtakes only once the penalty exceeds the computed crossover.
method #4 / #1 (reproducibility)
- a reproducibility-manifest cell prints seeds, sample sizes, all PSA distributions, structural priors and package versions; the regimen note states explicitly that the optimised re-dose interval is illustrative and does
not
enter the base-case economics.
v2 foundations (retained)
¶
P0-1
one shared exposure->closure model so mechanism drives the economics (v1 had two disconnected closure models with opposite drug rankings).
P0-2
budget-impact downstream saving derived from the Markov disease-cost stream, so BIA and CEA agree in sign.
P0-3
decision-level uncertainty: winner-flip thresholds, decision tornado, P(each strategy optimal), probabilistic EVIC, 2-D decision map - all in the subgroup-weighted lens, kept separate from the homogeneous average-patient reference case.
Two lenses are reported and kept distinct throughout:
a
homogeneous average-patient reference case
(efficient-frontier winner: chlorhexidine chip) and a
subgroup-prevalence-weighted target-population decision
. Numbers in v3 differ from v2 because the double count is removed and structural uncertainty is added.


In [ ]:
# =====================================================================================
# 0 - Environment, global style, and the single canonical parameter block (CFG)
# =====================================================================================
import os, json, warnings, sys
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.integrate import solve_ivp
from scipy.interpolate import PchipInterpolator
from scipy.optimize import minimize_scalar, brentq, least_squares
from scipy.stats import norm
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline

TRAPZ = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

RESULT = "reviewer_revision_results"; FIGDIR = os.path.join(RESULT, "figures"); TABDIR = os.path.join(RESULT, "tables")
for d in (RESULT, FIGDIR, TABDIR):
    os.makedirs(d, exist_ok=True)
# Remove superseded exports from the dedicated generated-output folders so
# each run contains only the current sequential index set.
for _folder, _suffix in ((FIGDIR, ".tiff"), (TABDIR, ".csv")):
    for _filename in os.listdir(_folder):
        if _filename.lower().endswith(_suffix):
            os.remove(os.path.join(_folder, _filename))

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 10, "font.family": "DejaVu Sans",
    "axes.titlesize": 10.5, "axes.labelsize": 10, "axes.linewidth": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "axes.axisbelow": True,
    "legend.fontsize": 8.5, "legend.frameon": False,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
})
C = dict(atridox="#2471A3", periochip="#B9770E", arestin="#7D3C98",
         srp="#5D6D7E", accent="#C0392B", ok="#1E8449", mic="#7F8C8D",
         warn="#E67E22", cool="#148F77", grid="#BDC3C7")
RNG_SEED = 2024
rng = np.random.default_rng(RNG_SEED)

_saved_tiffs = []         # ipynb builder hook: figures saved, in cell order
def save_fig(fig, name, close=True):
    tiff = os.path.join(FIGDIR, name + ".tiff")
    fig.savefig(tiff, dpi=300, format="tiff", bbox_inches="tight", facecolor="white")
    _saved_tiffs.append(tiff)
    if close: plt.close(fig)
    return tiff

def panel_tag(ax, s, dx=-0.02, dy=1.04, fs=12):
    ax.text(dx, dy, s, transform=ax.transAxes, fontsize=fs, fontweight="bold",
            va="bottom", ha="right")

# ------------------------------------------------------------------------------------
# CFG - ONE source of truth for every downstream parameter
# ------------------------------------------------------------------------------------
CFG = dict(
    ANALYSIS_CLASS="proof_of_concept_simulation_not_decision_grade",
    CLOSURE_MAPPING="linear",
    CLOSURE_MAPPINGS=("linear", "logistic", "probit", "emax"),
    CLOSURE_MAPPING_WEIGHTS="equal_scenario_weights_not_probabilities",
    WTP=50000.0, WTP_ALT=(20000.0, 30000.0, 50000.0, 100000.0),
    DISCOUNT=0.03, HORIZON_Y=20,
    UTIL=np.array([0.95, 0.88, 0.80, 0.70]),
    MAINT_COST=180.0,
    # closure prob = clip(P0_SRP + KAPPA_CLOSE * incremental PPD reduction[mm], floor, PCLOSE_CEIL)
    # P0_SRP, KAPPA_CLOSE and PCLOSE_CEIL are uncalibrated structural assumptions.
    # Aggregate PPD WMDs do not identify a pocket-closure probability; these values are
    # propagated only to demonstrate structural uncertainty, not as evidence-derived inputs.
    P0_SRP=0.33, KAPPA_CLOSE=0.25, PCLOSE_CEIL=0.75,
    # Illustrative placeholders; no payer/source attribution is claimed.
    COURSE_COST={"SRP alone": 0.0, "SRP+Atridox": 300.0,
                 "SRP+Periochip": 290.0, "SRP+Arestin": 330.0},
    # host-modulation (anti-MMP-8) incremental oral-QALY, Caton 2000 anchor
    HM_FACTOR={"SRP alone": 0.0, "SRP+Atridox": 1.0,
               "SRP+Periochip": 0.0, "SRP+Arestin": 0.4},
    HM_BASE_QALY=0.0,  # no empirical local-LDD -> QALY mapping; base case disabled
    SUBGROUP_PREV={"Non-smoker/shallow": 0.40, "Smoker/deep": 0.20,
                   "Diabetic/moderate": 0.25, "Smoker+diabetic": 0.15},
    SUBGROUP_CLOSURE_DELTA={"Non-smoker/shallow": +0.06, "Smoker/deep": -0.12,
                            "Diabetic/moderate": -0.07, "Smoker+diabetic": -0.16},
    CLOSURE_MARKOV_COUPLING="initial_only",
    CLOSURE_MARKOV_COUPLING_SCENARIOS=("initial_only", "transition_only", "dual"),
    MIC_DOX=6.0, MIC_MINO=1.0, CHX_BIOCIDAL=125.0,
    # antibiotic-stewardship penalty (per-course expected resistance cost, $) - base 0,
    # swept in the stewardship analysis; Caton 2000 found NO doxycycline resistance with SDD
    STEWARDSHIP_COST={"SRP alone": 0.0, "SRP+Atridox": 0.0,
                      "SRP+Periochip": 0.0, "SRP+Arestin": 0.0},
    # ---- reviewer-response switches (v3) -------------------------------------------------
    # #2 SINGLE host-modulation pathway (no double count): host modulation acts EITHER through
    #    improved pocket closure ("closure") OR through a separate anti-inflammatory oral-QALY
    #    increment ("utility") -- NEVER both. "none" removes it entirely (conservative base case).
    HM_PATHWAY="none",
    # #1 host modulation is an SDD-extrapolated SCENARIO (Caton 2000, sub-antimicrobial dosing) ->
    #    its STRENGTH is a scenario multiplier that is swept (none / weaker / current).
    HM_SCENARIO="none", HM_SCEN_MULT={"none": 0.0, "weaker": 0.5, "current": 1.0},
    # #6 / structural-uncertainty: propagate the closure-MAPPING structural parameters in the PSA.
    PROPAGATE_STRUCTURAL_PSA=True,
    STRUCT_PSA=dict(P0_SRP=(0.33, 0.05, 0.15, 0.50),     # (mean, sd, lo, hi) truncated-normal prior
                    KAPPA_CLOSE=(0.25, 0.06, 0.10, 0.45),
                    PCLOSE_CEIL=(0.75, 0.06, 0.60, 0.90)),
    GAMMA_CL_RANGE=(0.8, 1.8),   # Hill response-gradient, one-way structural check
    # #10 analysis unit: the Markov trace is ONE representative treated periodontal site (index
    #     tooth); reported cost/QALY outputs use N_INDEX_SITES treated site(s); the base case uses one treated site.
    N_INDEX_SITES=1,
)
print("Environment ready. numpy", np.__version__, "| pandas", pd.__version__, "| seed", RNG_SEED)
print("Base-case WTP = ${:,.0f}/QALY | discount {:.0%} | horizon {} y".format(
    CFG["WTP"], CFG["DISCOUNT"], CFG["HORIZON_Y"]))



In [ ]:
# =====================================================================================
# 1 - Evidence-audited multi-source evidence base
# =====================================================================================
# Lipophilicity is pH-dependent for ionisable tetracyclines. The values below are logD_oct
# at pH 6.5, not intrinsic logP and not the apparent partition coefficients 0.60/1.10 that
# were previously mislabelled. Source: Yamauchi, Inoue & Sugano, ADMET DMPK 2020;
# DOI 10.5599/admet.797, Table 1.
physchem = pd.DataFrame([
    ("Doxycycline",  -0.08, 444.43, "3.0/8.0/9.2", "active moiety", "Yamauchi 2020; DOI 10.5599/admet.797"),
    ("Minocycline",   0.20, 457.48, "2.8/5.0/7.8/9.5", "active moiety", "Yamauchi 2020; DOI 10.5599/admet.797"),
    ("Chlorhexidine gluconate", np.nan, 897.8, "strongly basic", "PerioChip salt form", "FDA PerioChip label 2011"),
], columns=["Drug", "LogD_pH6p5", "MW", "pKa", "Form_note", "Source"])

# Reported GCF summary anchors. These are study/group means or label summaries, not raw IPD.
# ATRIDOX 18-h value is a conservative lower-bound representation of "remained above 1000".
# ARESTIN public labeling supports a ~340 ug/mL 14-day plateau summary; it is not a digitized
# multi-time-point profile and is therefore not eligible for external-validation claims.
PK_ANCHORS = {
    "Atridox": dict(product="Atridox (DOX gel)", carrier="ATRIGEL polymer",
        dose_mg=42.5, dose_basis="total syringe; per-pocket mass not reported",
        threshold=CFG["MIC_DOX"], thr_type="label susceptibility threshold (<=6 ug/mL)", is_biocide=False,
        systemic="serum <=0.1 ug/mL in Stoller study", release="burst + sustained", t_end=7.0, label_days=7.0,
        t=[0, 2/24, 18/24, 7], c=[0, 1473, 1000, 309],
        data_kind="reported group summary; 18 h is conservative lower bound",
        src="Stoller et al. 1998, DOI 10.1902/jop.1998.69.10.1085; FDA ATRIDOX label"),
    "Periochip": dict(product="PerioChip (CHX gluconate chip)", carrier="cross-linked gelatin chip",
        dose_mg=2.5, dose_basis="one chip",
        threshold=CFG["CHX_BIOCIDAL"], thr_type="99% subgingival-bacteria inhibition level (FDA review)", is_biocide=True,
        systemic="plasma/urine undetectable", release="biphasic", t_end=9.0, label_days=9.0,
        t=[0, 2/24, 4/24, 3, 9], c=[0, 2007, 1444, 1902, 57],
        data_kind="reported group means",
        src="Soskolne et al. 1998, DOI 10.1111/j.1600-051x.1998.tb02407.x; FDA PerioChip label"),
    "Arestin": dict(product="Arestin (MINO microspheres)", carrier="PGLA microspheres",
        dose_mg=1.0, dose_basis="one unit-dose cartridge/site",
        threshold=CFG["MIC_MINO"], thr_type="FDA review efficacy threshold (>1 ug/mL)", is_biocide=False,
        systemic="dose-normalized serum Cmax 0.00488 ug/mL per mg (mean full-mouth dose 46.2 mg)",
        release="label-summary plateau", t_end=14.0, label_days=14.0,
        t=[0, 1/24, 14], c=[0, 340, 340],
        data_kind="label-summary plateau representation; not digitized time series",
        src="FDA NDA 50-781 clinical-pharmacology review; ARESTIN label"),
}

# Clinical effects. Atridox/Arestin values and CIs are product-stratified 6-9 month WMDs
# from Soysa et al. 2025. PerioChip uses Ma & Diao 2020 at 6 months; their article reports
# point estimates in text but the earlier notebook did not preserve a defensible CI. The
# 0.30-0.75 range is therefore an evidence-scenario interval (FDA pivotal 9-month increment
# to Ma-Diao 6-month meta-analysis), NOT a 95% CI.
clinical = pd.DataFrame([
    ("SRP alone",     0.000, 0.000, 0.000, 0.000, None, "reference", "not applicable", "control"),
    ("SRP+Atridox",   0.446, 0.029, 0.862, 0.495, 4, "95% CI", "product-stratified meta-analysis",
     "Soysa et al. 2025; DOI 10.3389/fdmed.2025.1658720"),
    ("SRP+Periochip", 0.750, 0.300, 0.750, 0.680, 15, "evidence scenario", "6-month meta-analysis; lower point from FDA pivotal 9-month trials",
     "Ma & Diao 2020; DOI 10.1186/s12903-020-01247-8; FDA PerioChip label"),
    ("SRP+Arestin",   0.406, 0.279, 0.534, 0.298, 17, "95% CI", "product-stratified meta-analysis",
     "Soysa et al. 2025; DOI 10.3389/fdmed.2025.1658720"),
], columns=["Strategy", "PPD_WMD", "PPD_lo", "PPD_hi", "CAL_WMD", "k_studies", "Range_type", "Evidence_note", "Source"])

def _bounded_link_calibration(p0, kappa, floor, ceil, link):
    """Return intercept/slope so a bounded link has value p0 and derivative kappa at x=0."""
    span = float(ceil - floor)
    q0 = float(np.clip((p0 - floor) / span, 1e-6, 1 - 1e-6))
    if link == "logistic":
        intercept = float(np.log(q0 / (1 - q0)))
        slope = float(kappa / (span * q0 * (1 - q0)))
    elif link == "probit":
        intercept = float(norm.ppf(q0))
        slope = float(kappa / (span * norm.pdf(intercept)))
    else:
        raise ValueError(f"Unknown bounded link: {link}")
    return intercept, slope


def closure_prob(ppd_wmd, p0=None, kappa=None, ceil=None, floor=0.05, mapping=None):
    """Structural PPD-to-closure transform used only for proof-of-concept propagation.

    Aggregate PPD WMD does not identify a patient/site-level pocket-closure probability.
    Linear, logistic, probit and Emax alternatives are evaluated as structural scenarios.
    The nonlinear links match the same p0 and local slope at zero so their comparison isolates
    functional-form uncertainty. All mappings are monotone and therefore cannot independently
    validate a product ranking already present in the PPD-effect inputs.
    """
    p0 = CFG["P0_SRP"] if p0 is None else float(p0)
    kappa = CFG["KAPPA_CLOSE"] if kappa is None else float(kappa)
    ceil = CFG["PCLOSE_CEIL"] if ceil is None else float(ceil)
    mapping = CFG["CLOSURE_MAPPING"] if mapping is None else str(mapping).lower()
    x = max(float(ppd_wmd), 0.0)
    if mapping == "linear":
        value = p0 + kappa * x
    elif mapping in ("logistic", "probit"):
        intercept, slope = _bounded_link_calibration(p0, kappa, floor, ceil, mapping)
        z = intercept + slope * x
        linked = 1.0 / (1.0 + np.exp(-z)) if mapping == "logistic" else norm.cdf(z)
        value = floor + (ceil - floor) * linked
    elif mapping == "emax":
        ec50 = max((ceil - p0) / max(kappa, 1e-9), 1e-9)
        value = p0 + (ceil - p0) * x / (ec50 + x)
    else:
        raise ValueError(f"Unknown closure mapping: {mapping}")
    return float(np.clip(value, floor, ceil))

RESP = {row.Strategy: closure_prob(row.PPD_WMD) for row in clinical.itertuples()}
clinical["ClosureProxy"] = clinical["Strategy"].map(RESP)
clinical["ClosureProb"] = clinical["ClosureProxy"]  # compatibility with downstream export

PARAMETER_PROVENANCE = pd.DataFrame([
    ("LogD_pH6.5 doxycycline", -0.08, "reported", "Yamauchi 2020", "10.5599/admet.797"),
    ("LogD_pH6.5 minocycline", 0.20, "reported", "Yamauchi 2020", "10.5599/admet.797"),
    ("Doxycycline threshold", 6.0, "FDA label", "ATRIDOX label", "NDA 50-751"),
    ("Minocycline threshold", 1.0, "FDA review", "ARESTIN clinical pharmacology", "NDA 50-781"),
    ("CHX 99% inhibition level", 125.0, "FDA review", "PerioChip clinical review", "NDA 20-774"),
    ("PPD-to-closure p0", CFG["P0_SRP"], "structural assumption", "none", "not empirical"),
    ("PPD-to-closure slope", CFG["KAPPA_CLOSE"], "structural assumption", "none", "not empirical"),
    ("PPD-to-closure ceiling", CFG["PCLOSE_CEIL"], "structural assumption", "none", "not empirical"),
    ("PPD-to-closure functional forms", "linear/logistic/probit/emax", "structural scenarios", "reviewer-requested sensitivity", "not empirical"),
], columns=["Parameter", "Value", "Evidence_class", "Source", "Identifier"])

assert CFG["MIC_MINO"] == 1.0
assert clinical.loc[clinical.Strategy.eq("SRP+Periochip"), "Range_type"].iat[0] == "evidence scenario"
print("Physicochemical descriptors (pH-specific logD, not mislabelled logP):")
print(physchem.to_string(index=False))
print("\nClinical effects and STRUCTURAL closure proxy:")
print(clinical[["Strategy", "PPD_WMD", "PPD_lo", "PPD_hi", "Range_type", "CAL_WMD", "ClosureProxy", "Source"]].to_string(index=False))
print("\nIMPORTANT: ClosureProxy, recurrence and economic outputs are model assumptions, not measured outcomes.")




In [ ]:
# PK/PD indices from reported summary-anchor representations (CHX index labelled as biocidal, not MIC).
def make_profile(t_pts, c_pts, t_end):
    t = np.asarray(t_pts, float); c = np.asarray(c_pts, float)
    pch = PchipInterpolator(t, c)
    def f(x):
        x = np.atleast_1d(np.asarray(x, float))
        return np.clip(np.where(x <= t_end, pch(np.clip(x, t[0], t[-1])), 0.0), 0, None)
    return f
profiles = {k: make_profile(v["t"], v["c"], v["t_end"]) for k, v in PK_ANCHORS.items()}

def pkpd_indices(name):
    spec = PK_ANCHORS[name]; f = profiles[name]; thr = spec["threshold"]
    x = np.linspace(0, spec["t_end"], 8000); c = f(x)
    cmax = c.max(); auc = TRAPZ(c, x)
    above = x[c >= thr]
    t_above = float(above.max() - above.min()) if len(above) else 0.0
    idx_name = "T>biocidal_d" if spec["is_biocide"] else "T>MIC_d"
    return dict(Product=spec["product"], Cmax=cmax, Threshold=thr, Biocide=spec["is_biocide"],
                Cmax_thr=cmax/thr, AUC_thr=auc/thr, Tabove_d=t_above, pctTabove=100*(c >= thr).mean(),
                idx_name=idx_name)

PK_ANCHOR_KEYS = list(PK_ANCHORS.keys())
pkpd_tbl = pd.DataFrame([pkpd_indices(k) for k in PK_ANCHOR_KEYS]).round(1)
print("\nExploratory PK/PD indices from summary-anchor representations (CHX = time above BIOCIDAL threshold, not MIC):")
print(pkpd_tbl[["Product","Cmax","Threshold","Cmax_thr","AUC_thr","Tabove_d","pctTabove"]].to_string(index=False))

# label exposure (days above threshold) per product - the anchor point on the shared Hill
TABOVE = {k: pkpd_indices(k)["Tabove_d"] for k in PK_ANCHOR_KEYS}
STRAT_OF = {"Atridox": "SRP+Atridox", "Periochip": "SRP+Periochip", "Arestin": "SRP+Arestin"}
print("\nLabel time-above-threshold (days):", {k: round(v,1) for k,v in TABOVE.items()})



In [ ]:
# ---- FIGURE 1: standardized evidence base + PK/PD indices ----
fig1 = plt.figure(figsize=(13.5, 4.3))
gs = GridSpec(1, 3, figure=fig1, width_ratios=[1.25, 1.0, 1.1], wspace=0.32)
prod_col = {"Atridox": C["atridox"], "Periochip": C["periochip"], "Arestin": C["arestin"]}

axA = fig1.add_subplot(gs[0, 0])
tt = np.linspace(0, 14.5, 800)
for k, spec in PK_ANCHORS.items():
    axA.plot(tt, np.clip(profiles[k](tt), 1e-3, None), color=prod_col[k], lw=2.2, label=spec["product"])
    axA.scatter(spec["t"], np.clip(spec["c"], 1e-3, None), color=prod_col[k], s=16, zorder=5)
axA.set_yscale("log"); axA.set_ylim(1, 3000)
axA.set_xlabel("Time (days)"); axA.set_ylabel("GCF concentration (µg/mL, log)")
axA.set_title("Digitized GCF PK profiles"); axA.legend(loc="upper right", fontsize=7.5)
panel_tag(axA, "A")

axB = fig1.add_subplot(gs[0, 1])
layers = ["Physchem", "PK profile", "Threshold", "Systemic", "Meta WMD", "Pivotal RCT", "Inflamm."]
prods = ["Atridox", "Periochip", "Arestin"]
# 1 = full evidence, 0.5 = partial/indirect, marked with a ring
M = np.array([[1,1,1,1,1,1,1],[0.5,1,1,1,1,1,0.5],[1,1,1,1,1,1,1]], float)
axB.imshow(M, cmap="Greens", vmin=0, vmax=1.4, aspect="auto")
for i in range(3):
    for j in range(7):
        if M[i,j] == 1: axB.text(j, i, "✓", ha="center", va="center", fontsize=9)
        elif M[i,j] == 0.5: axB.text(j, i, "~" if j==0 else "○", ha="center", va="center", fontsize=9)
axB.set_xticks(range(7)); axB.set_xticklabels(layers, rotation=45, ha="right", fontsize=7.5)
axB.set_yticks(range(3)); axB.set_yticklabels(prods)
axB.set_title("Evidence-availability matrix"); axB.grid(False); panel_tag(axB, "B")

axC = fig1.add_subplot(gs[0, 2])
xlab = ["Atridox", "Periochip", "Arestin"]; xpos = np.arange(3); w = 0.26
cmax_thr = [pkpd_indices(k)["Cmax_thr"] for k in xlab]
auc_thr  = [pkpd_indices(k)["AUC_thr"] for k in xlab]
tabove   = [pkpd_indices(k)["Tabove_d"] for k in xlab]
axC.bar(xpos-w, cmax_thr, w, color=C["accent"], label="Cmax/threshold")
axC.bar(xpos,   auc_thr,  w, color=C["cool"],   label="AUC/threshold")
axC.bar(xpos+w, tabove,   w, color=C["warn"],   label="T>threshold (d)")
axC.set_yscale("log"); axC.set_xticks(xpos); axC.set_xticklabels(xlab)
axC.set_ylabel("Index value (log)"); axC.set_title("Pharmacometric PK/PD indices")
axC.legend(fontsize=7.5); panel_tag(axC, "C")

fig1.suptitle("Figure 1  |  Stage 1: standardized multi-source evidence base and PK/PD indices",
              y=1.02, fontsize=12, fontweight="bold", ha="left", x=0.02)
save_fig(fig1, "Figure01_data_and_PKPD_indices")
print("Saved Figure 1")



In [ ]:
# =====================================================================================
# 2 - ML feature/parameter screening recovers the encoded mechanisms (self-consistency)
#   Synthetic software-unit-test cohort; not patient data and not an evidence analysis.
# =====================================================================================
N = 500
logp     = rng.normal(0.06, 0.25, N)     # synthetic logD(pH 6.5), centred between -0.08 and 0.20
mw       = rng.normal(500, 120, N)      # molecular weight (noise feature)
release  = rng.uniform(5, 15, N)        # formulation retention (days)
ppd      = rng.uniform(4, 9, N)         # baseline pocket depth (mm)
gcf_flow = rng.uniform(0.5, 12.0, N)    # synthetic effective clearance (uL/h); not measured flow
smoking  = rng.integers(0, 2, N)
diabetes = rng.integers(0, 2, N)
age      = rng.normal(52, 12, N)
particle = rng.uniform(20, 60, N)       # microsphere size proxy (um)

tmic = np.clip(0.75*release + 1.2*logp - 0.15*gcf_flow - 0.35*(ppd-4)
               + 0.02*(particle-40) + 5.0 + rng.normal(0, 0.6, N), 0.5, 16)
resp = 1/(1+np.exp(-((0.10*tmic - 0.8*smoking - 0.5*diabetes - 0.03*(ppd-5)
                      + 0.4*logp + rng.normal(0, 0.4, N)) - 1.0)))
X = pd.DataFrame({"LogD_pH6p5": logp, "MolWeight": mw, "ReleaseDays": release, "BaselinePPD": ppd,
                  "GCF_flow": gcf_flow, "Smoking": smoking, "Diabetes": diabetes,
                  "Age": age, "ParticleSize": particle})

Xs = StandardScaler().fit_transform(X)
lasso = LassoCV(cv=5, random_state=RNG_SEED, max_iter=20000).fit(Xs, tmic)
lasso_r2 = cross_val_score(LassoCV(cv=5, random_state=RNG_SEED, max_iter=20000), Xs, tmic, cv=5).mean()
lasso_coef = pd.Series(lasso.coef_, index=X.columns).sort_values(key=np.abs, ascending=False)

rf = RandomForestRegressor(n_estimators=400, random_state=RNG_SEED, n_jobs=1).fit(X, resp)
rf_r2 = cross_val_score(RandomForestRegressor(n_estimators=200, random_state=RNG_SEED, n_jobs=1),
                        X, resp, cv=5).mean()
perm = permutation_importance(rf, X, resp, n_repeats=20, random_state=RNG_SEED, n_jobs=1)
rf_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

top_tmic = list(lasso_coef.index[:3]); top_resp = list(rf_imp.index[:3])
print(f"LASSO (T>MIC) CV R2 = {lasso_r2:.3f}   |   RandomForest (response) CV R2 = {rf_r2:.3f}")
print("Top drivers of T>MIC   :", top_tmic)
print("Top drivers of response:", top_resp)
print("-> ML recovers the encoded mechanisms (release+lipophilicity drive exposure; "
      "lipophilicity+host factors drive response). This is a self-consistency check, NOT evidence the mechanism is true.")



In [ ]:
# ---- FIGURE 2: ML feature/parameter screening ----
fig2 = plt.figure(figsize=(13.5, 4.4))
gs = GridSpec(1, 3, figure=fig2, width_ratios=[1, 1, 1], wspace=0.42)

axA = fig2.add_subplot(gs[0, 0])
cc = lasso_coef.sort_values()
axA.barh(range(len(cc)), cc.values,
         color=[C["ok"] if v > 0 else C["accent"] for v in cc.values])
axA.axvline(0, color="k", lw=0.8); axA.set_yticks(range(len(cc))); axA.set_yticklabels(cc.index)
axA.set_xlabel("LASSO standardized coefficient")
axA.set_title(f"Drivers of local exposure T>MIC (CV R²={lasso_r2:.2f})"); panel_tag(axA, "A")

axB = fig2.add_subplot(gs[0, 1])
ii = rf_imp.sort_values()
axB.barh(range(len(ii)), ii.values, color=C["periochip"])
axB.set_yticks(range(len(ii))); axB.set_yticklabels(ii.index)
axB.set_xlabel("Permutation importance (ΔR²)")
axB.set_title(f"Drivers of clinical response (RF, CV R²={rf_r2:.2f})"); panel_tag(axB, "B")

axC = fig2.add_subplot(gs[0, 2])
pred_tmic = lasso.predict(Xs)
axC.scatter(tmic, pred_tmic, s=10, alpha=0.4, color=C["atridox"])
lim = [tmic.min()-0.5, tmic.max()+0.5]; axC.plot(lim, lim, "--", color=C["mic"], lw=1)
axC.set_xlabel("True T>MIC (days)"); axC.set_ylabel("LASSO-predicted T>MIC")
rmse = float(np.sqrt(np.mean((tmic-pred_tmic)**2)))
axC.set_title(f"Mechanism recovery (RMSE={rmse:.2f} d)"); panel_tag(axC, "C")

fig2.suptitle("Figure 2  |  Stage 2: ML feature/parameter screening recovers encoded mechanisms",
              y=1.02, fontsize=12, fontweight="bold", ha="left", x=0.02)
save_fig(fig2, "Figure02_ML_feature_selection")
print("Saved Figure 2")



In [ ]:
# =====================================================================================
# 3 - Mechanistic local mini-PBPK: dynamics, calibration checks, Sobol GSA
#   volumes in uL, amounts in ug, time in h
# =====================================================================================
# R0 is derived from 1 mg / 14 d. CL_eff (~8.75, rounded 9 uL/h) is back-calculated
# from R0 / the ~340 ug/mL label plateau and is therefore a calibrated effective clearance,
# not a direct Goodson flow measurement. Biofilm/tissue parameters are non-identifiable
# nuisance parameters. The fifth state is cumulative absorbed mass, not a human plasma PK
# compartment; no serum concentration or label-derived absorbed fraction is claimed.
PBPK = dict(
    R0=1000.0/(14*24), Vpf=0.5, Vbf=0.3, Vti=20.0,
    CL_eff=9.0, PSbf=1.2, PSti=2.0, Kp=2.2,
    kbf=0.006, kabs=0.015, delta=1.0,
)
def pbpk_rhs(t, y, p):
    A_dep, A_pf, A_bf, A_ti, A_abs = y
    R = p["R0"] * 0.5 * (1.0 + np.tanh(A_dep / p["delta"]))
    C_pf, C_bf, C_ti = A_pf/p["Vpf"], A_bf/p["Vbf"], A_ti/p["Vti"]
    J_bf = p["PSbf"] * (C_pf - C_bf)
    J_ti = p["PSti"] * (C_pf - C_ti/p["Kp"])
    W = p["CL_eff"] * C_pf
    return [-R, R - W - J_bf - J_ti, J_bf - p["kbf"]*A_bf,
            J_ti - p["kabs"]*A_ti, p["kabs"]*A_ti]

def run_pbpk(p, dose=1000.0, days=18, n=4000, rtol=1e-8, atol=1e-10):
    te = np.linspace(0, 24*days, n)
    return solve_ivp(pbpk_rhs, (0, 24*days), [dose, 0, 0, 0, 0], t_eval=te,
                     args=(p,), method="LSODA", rtol=rtol, atol=atol)

with warnings.catch_warnings():
    warnings.simplefilter("error")
    sol = run_pbpk(PBPK)
assert sol.success
tt = sol.t; td = tt/24; A = sol.y
Cpf = A[1]/PBPK["Vpf"]*1000; Cbf = A[2]/PBPK["Vbf"]*1000; Cti = A[3]/PBPK["Vti"]*1000
MIC = CFG["MIC_MINO"]
plateau = Cpf[(td >= 1) & (td <= 14)].mean()
tmic_bf = td[Cbf >= MIC].max() if (Cbf >= MIC).any() else 0.0
sys_pct = 100*A[4, -1]/1000.0
W_cum = TRAPZ(PBPK["CL_eff"]*A[1]/PBPK["Vpf"], tt)
bf_cum = TRAPZ(PBPK["kbf"]*A[2], tt)
remaining = A[:, -1].sum()
mb_err = abs((remaining + W_cum + bf_cum) - 1000)/1000*100
print("[Local mini-PBPK - ARESTIN label-summary paradigm]")
print(f"  pocket plateau ~{plateau:.0f} ug/mL (published summary ~340 ug/mL through day 14)")
print(f"  tissue peak {Cti.max():.0f} ug/mL (calibrated Kp={PBPK['Kp']})")
print(f"  T>1 ug/mL in biofilm {tmic_bf:.1f} d")
print(f"  model absorbed-sink fraction {sys_pct:.1f}% (NOT a label value or validated bioavailability)")
print(f"  MASS-BALANCE error = {mb_err:.3f}%  (solver: LSODA)")




In [ ]:
# ---- Three dosage-form calibration to published GCF summary anchors ----
# This is calibration/goodness-of-fit, not external validation. PerioChip (2.5 mg) and
# ARESTIN (1 mg/site) use fixed known doses. ATRIDOX's 42.5 mg is total syringe content;
# because the treated-side/per-pocket mass is not reported, its effective input is a bounded
# nuisance scale (0.1-42.5 mg) and is never interpreted as an estimated dose.
def pbpk_rhs_gen(t, y, p):
    A_d1, A_d2, A_pf, A_bf, A_ti, A_abs = y
    if p["rel_type"] == "zero":
        R = p["R0"]*0.5*(1.0+np.tanh(A_d1/p["delta"])); dD1, dD2 = -R, 0.0
    else:
        R = p["krel1"]*A_d1 + p["krel2"]*A_d2
        dD1, dD2 = -p["krel1"]*A_d1, -p["krel2"]*A_d2
    C_pf, C_bf, C_ti = A_pf/p["Vpf"], A_bf/p["Vbf"], A_ti/p["Vti"]
    J_bf = p["PSbf"]*(C_pf-C_bf); J_ti = p["PSti"]*(C_pf-C_ti/p["Kp"])
    W = p["CL_eff"]*C_pf
    return [dD1, dD2, R-W-J_bf-J_ti, J_bf-p["kbf"]*A_bf,
            J_ti-p["kabs"]*A_ti, p["kabs"]*A_ti]

def run_gen(p, dose, tmax_d, teval_d):
    ff = float(p.get("f_fast", 0.6))
    y0 = ([ff*dose, (1-ff)*dose, 0, 0, 0, 0] if p["rel_type"] == "biphasic"
          else [dose, 0, 0, 0, 0, 0])
    te = np.clip(np.asarray(teval_d, float)*24.0, 0, tmax_d*24)
    s = solve_ivp(pbpk_rhs_gen, (0, tmax_d*24), y0, t_eval=te, args=(p,),
                  method="LSODA", rtol=1e-7, atol=1e-9)
    if not s.success: raise RuntimeError(s.message)
    return s

def pocket_curve_gen(p, dose, tmax_d, teval_d):
    s = run_gen(p, dose, tmax_d, teval_d)
    return s.y[2]/p["Vpf"]*1000.0

REL_TYPE = {"Atridox": "biphasic", "Periochip": "biphasic", "Arestin": "zero"}
def _sigmoid(z): return 1.0/(1.0+np.exp(-z))

def fit_product(name, n_starts=12):
    spec = PK_ANCHORS[name]; rtype = REL_TYPE[name]
    t = np.asarray(spec["t"], float); c = np.asarray(spec["c"], float)
    keep = t > 0; t, c = t[keep], c[keep]
    order = np.argsort(t); t, c = t[order], c[order]
    rel = spec["label_days"]; tmax = rel + 4
    base = dict(PBPK, rel_type=rtype)
    fit = None; fitted_names = []; theta = np.array([]); jac_singular = np.array([])
    jac_rank = 0; condition_number = np.nan; near_count = 0; near_span = np.nan
    if rtype == "zero":
        dose = spec["dose_mg"]*1000.0
        p = dict(base, R0=dose/(rel*24.0))
        dose_note = "fixed label-summary dose/release; no parameters estimated"
        identifiability = "not assessed: zero-order representation fixed from two label-summary anchors"
    else:
        fitted_names = ["log_kfast", "log_kslow", "logit_fast_fraction"]
        if name == "Atridox": fitted_names.append("log_effective_input")
        x0 = np.array([np.log(0.2), np.log(0.005), 0.0] + ([np.log(3000.0)] if name == "Atridox" else []))
        lower = np.array([np.log(1e-3), np.log(1e-5), -7.0] + ([np.log(100.0)] if name == "Atridox" else []))
        upper = np.array([np.log(10.0), np.log(1.0), 7.0] + ([np.log(42500.0)] if name == "Atridox" else []))

        def unpack(candidate):
            parameters = dict(base, krel1=float(np.exp(candidate[0])),
                              krel2=float(np.exp(candidate[1])), f_fast=float(_sigmoid(candidate[2])))
            effective_dose = float(np.exp(candidate[3])) if name == "Atridox" else spec["dose_mg"]*1000.0
            return parameters, effective_dose

        def residual(candidate):
            parameters, effective_dose = unpack(candidate)
            prediction = pocket_curve_gen(parameters, effective_dose, tmax, t)
            return np.log(np.clip(prediction, 1e-3, None)) - np.log(np.clip(c, 1e-3, None))

        fit_rng = np.random.default_rng(RNG_SEED + sum(ord(ch) for ch in name))
        starts = [x0]
        for _ in range(n_starts - 1):
            starts.append(fit_rng.uniform(lower + 1e-6, upper - 1e-6))
        candidates = []
        for start in starts:
            try:
                candidate = least_squares(residual, start, bounds=(lower, upper), method="trf", max_nfev=1800)
                if candidate.success and np.isfinite(candidate.cost):
                    candidates.append(candidate)
            except (FloatingPointError, RuntimeError, ValueError):
                continue
        if not candidates:
            raise RuntimeError(f"All calibration starts failed for {name}")
        fit = min(candidates, key=lambda item: item.cost)
        theta = fit.x.copy(); p, dose = unpack(theta)
        dose_note = "bounded nuisance input; not per-pocket dose" if name == "Atridox" else "fixed label dose"
        jacobian = np.asarray(fit.jac, float)
        jac_singular = np.linalg.svd(jacobian, compute_uv=False) if jacobian.size else np.array([])
        jac_rank = int(np.linalg.matrix_rank(jacobian)) if jacobian.size else 0
        condition_number = (float(jac_singular[0] / jac_singular[-1])
                            if jac_singular.size and jac_singular[-1] > 1e-12 else np.inf)
        tolerance = max(1e-7, 0.01 * max(float(fit.cost), 1e-7))
        near = [candidate.x for candidate in candidates if candidate.cost <= fit.cost + tolerance]
        near_count = len(near)
        near_span = float(np.max(np.ptp(np.vstack(near), axis=0))) if len(near) > 1 else 0.0
        n_obs, n_par = len(t), len(theta)
        if n_obs <= n_par:
            identifiability = "not identifiable: observations <= estimated parameters"
        elif jac_rank < n_par:
            identifiability = "not locally identifiable: rank-deficient residual Jacobian"
        elif not np.isfinite(condition_number) or condition_number > 1e4:
            identifiability = "weak practical identifiability: ill-conditioned residual Jacobian"
        elif near_span > 0.5:
            identifiability = "weak practical identifiability: near-optimal multistart solutions diverge"
        else:
            identifiability = "locally identifiable, but precision remains weak because anchors are sparse"
    pred = pocket_curve_gen(p, dose, tmax, t)
    aafe = float(np.exp(np.mean(np.abs(np.log(np.clip(pred,1e-3,None)/np.clip(c,1e-3,None))))))
    sink_end = run_gen(p, dose, tmax, [tmax]).y[5, -1]
    absorbed_pct = float(100.0*sink_end/dose)
    return dict(p=p, dose=dose, dose_note=dose_note, rel=rel, tmax=tmax, aafe_full=aafe,
                aafe_hold=np.nan, n_cal=len(t), n_tot=len(t), t=t, c=c, t_hold=np.array([]),
                rtype=rtype, f_fast=p.get("f_fast", np.nan), data_kind=spec["data_kind"],
                absorbed_pct=absorbed_pct, fitted_names=fitted_names, theta=theta,
                n_parameters=len(theta), jacobian_rank=jac_rank,
                condition_number=condition_number, jacobian_singular_values=jac_singular,
                near_optimal_solutions=near_count, near_optimal_max_span=near_span,
                identifiability=identifiability, independent_validation="not performed")


prod_fits = {k: fit_product(k) for k in PK_ANCHOR_KEYS}
assert np.isclose(prod_fits["Periochip"]["dose"], 2500.0)
assert np.isclose(prod_fits["Arestin"]["dose"], 1000.0)
assert 100.0 <= prod_fits["Atridox"]["dose"] <= 42500.0
_core = {"zero": "zero-order label summary", "biphasic": "biphasic burst+sustained"}
print("\nCalibration to published GCF summary anchors (not external validation):")
for k in PK_ANCHOR_KEYS:
    f = prod_fits[k]
    print(f"  {k:10s} [{_core[f['rtype']]:28s}]: calibration AAFE={f['aafe_full']:.2f}; "
          f"input={f['dose']/1000:.3g} mg ({f['dose_note']}); {f['data_kind']}")
print("  -> Sparse aggregate anchors do not support patient-level external validation or unique "
      "identification of depot, permeability, tissue, and absorption parameters.")

identifiability_tbl = pd.DataFrame([
    {
        "Product": PK_ANCHORS[key]["product"],
        "GCF anchors": fit["n_tot"],
        "Estimated release parameters": fit["n_parameters"],
        "Residual-Jacobian rank": fit["jacobian_rank"],
        "Condition number": (f"{fit['condition_number']:.2e}" if np.isfinite(fit["condition_number"]) else "not estimable"),
        "Near-optimal multistart solutions": fit["near_optimal_solutions"],
        "Maximum transformed-parameter span": (f"{fit['near_optimal_max_span']:.3f}"
                                                  if np.isfinite(fit["near_optimal_max_span"]) else "not applicable"),
        "Conclusion": fit["identifiability"],
        "Independent validation": fit["independent_validation"],
    }
    for key, fit in prod_fits.items()
])
print("\nRelease-parameter identifiability diagnostics:")
print(identifiability_tbl.to_string(index=False))
print("  -> These diagnostics concern only the release representation. The full local PBPK parameter set")
print("     remains non-identifiable from sparse pocket-fluid summary anchors.")




In [ ]:
# ---- Sobol GSA for absorbed-sink fraction and T>threshold ----
GSA_PARAMS = {
    "R0": (PBPK["R0"]*0.6, PBPK["R0"]*1.4), "CL_eff": (5.0, 14.0), "Kp": (1.2, 3.5),
    "kabs": (0.008, 0.025), "PSti": (1.0, 3.5), "Vpf": (0.3, 0.9),
}
gsa_names = list(GSA_PARAMS.keys()); d = len(gsa_names)
lows = np.array([GSA_PARAMS[k][0] for k in gsa_names]); highs = np.array([GSA_PARAMS[k][1] for k in gsa_names])
def gsa_outputs(theta):
    p = dict(PBPK, **{gsa_names[i]: theta[i] for i in range(d)})
    s = solve_ivp(pbpk_rhs, (0, 24*16), [1000, 0, 0, 0, 0], t_eval=np.linspace(0, 24*16, 700),
                  args=(p,), method="LSODA", rtol=1e-6, atol=1e-8)
    a = s.y; t = s.t/24; cbf = a[2]/p["Vbf"]*1000
    sysp = 100*a[4, -1]/1000.0
    tmic = t[cbf >= CFG["MIC_MINO"]].max() if (cbf >= CFG["MIC_MINO"]).any() else 0.0
    return np.array([sysp, tmic])
def saltelli_sobol(n_base=1024, seed=7, n_boot=300):
    r = np.random.default_rng(seed)
    Amat = lows + (highs-lows)*r.random((n_base, d)); Bmat = lows + (highs-lows)*r.random((n_base, d))
    fA = np.array([gsa_outputs(x) for x in Amat]); fB = np.array([gsa_outputs(x) for x in Bmat])
    n_out = fA.shape[1]; fAB = np.zeros((d, n_base, n_out))
    for i in range(d):
        AB = Amat.copy(); AB[:, i] = Bmat[:, i]; fAB[i] = np.array([gsa_outputs(x) for x in AB])
    varY = np.var(np.vstack([fA, fB]), axis=0)
    def estimate(idx):
        fAi, fBi, fABi = fA[idx], fB[idx], fAB[:, idx, :]
        s1 = 1.0 - 0.5*np.mean((fBi[None] - fABi)**2, axis=1)/varY
        st = 0.5*np.mean((fAi[None] - fABi)**2, axis=1)/varY
        return s1, st
    S1, ST = estimate(np.arange(n_base))
    S1b = np.zeros((n_boot, d, n_out)); STb = np.zeros((n_boot, d, n_out))
    for b in range(n_boot):
        bi = r.integers(0, n_base, n_base); S1b[b], STb[b] = estimate(bi)
    return np.clip(S1,0,1), np.clip(ST,0,1), np.percentile(S1b,[2.5,97.5],axis=0), np.percentile(STb,[2.5,97.5],axis=0)
print("\nRunning Sobol GSA (n_base=1024 + bootstrap; ~8192 PBPK solves)...")
S1, ST, S1ci, ST_ci = saltelli_sobol(n_base=1024)
sobol_sys = pd.DataFrame({"param": gsa_names, "S1": S1[:, 0], "ST": ST[:, 0]}).sort_values("ST", ascending=False)
sobol_tmic = pd.DataFrame({"param": gsa_names, "S1": S1[:, 1], "ST": ST[:, 1]}).sort_values("ST", ascending=False)
xover_sig = float(np.max(np.clip(S1ci[0] - ST_ci[1], 0, None)))
print(f"Sobol ABSORBED-SINK FRACTION - S1 sum = {S1[:,0].sum():.2f} (target <=1):")
print(sobol_sys.round(3).to_string(index=False))
print("Sobol T>MIC (days):"); print(sobol_tmic.round(3).to_string(index=False))
print(f"Max significant S1>ST crossover (CI-based) = {xover_sig:.3f} (0 => additive within MC error).")



In [ ]:
# =====================================================================================
# 3.5 - P0-1 CORE FIX: ONE unified exposure -> closure model (mechanism drives economics)
# -------------------------------------------------------------------------------------
# v1 had TWO closure models with opposite rankings: an evidence map (drove economics,
# Periochip>Atridox>Arestin) and a standalone T>MIC-only Emax (drove Fig4/5, Arestin best),
# and the Emax never touched the economics. Here all products sit on a SINGLE shared
# exposure->closure Hill. Each product is anchored at its meta-analytic evidence closure at
# LABEL exposure (evidence sets the level); the Hill provides the mechanistic RESPONSE
# GRADIENT (how closure moves when PBPK exposure changes via re-dosing / retention / PSA).
# This unifies the two representations and lets the regimen optimiser feed the economics.
E0 = CFG["P0_SRP"]; CEIL = CFG["PCLOSE_CEIL"]; GAMMA_CL = 1.2; X50 = 1.0
def hill(x, x50=X50, g=GAMMA_CL):
    x = np.asarray(x, float); return x**g/(x50**g + x**g)
def x_of_closure(pc, g=GAMMA_CL):
    frac = np.clip((np.asarray(pc,float)-E0)/(CEIL-E0), 1e-4, 0.999)
    return X50*(frac/(1-frac))**(1.0/g)
XLAB = {STRAT_OF[k]: float(x_of_closure(RESP[STRAT_OF[k]])) for k in PK_ANCHOR_KEYS}
def p_close_mult(strat, m, g=GAMMA_CL):
    """Closure at exposure multiplier m (m=1 -> exact evidence closure RESP[strat]).
    The Hill response-gradient g (GAMMA_CL) is exposed for a one-way structural check
    (reviewer #6); m=1 reproduces the evidence-anchored label closure for ANY g, so g only
    governs how closure moves when EXPOSURE changes (re-dosing / PSA), not the label levels."""
    xm = XLAB[strat]*np.asarray(m, float)
    val = E0 + (CEIL-E0)*(xm**g/(X50**g + xm**g))
    return float(val) if np.ndim(m)==0 else val
KEY_OF = {v: k for k, v in STRAT_OF.items()}
def p_close_exposure(strat, tabove_days):
    """Closure from an absolute time-above-threshold (days) for that product."""
    m = np.asarray(tabove_days, float)/TABOVE[KEY_OF[strat]]
    return p_close_mult(strat, m)
# verify anchoring: closure at label multiplier == evidence closure
print("Unified closure model (shared Hill, product-anchored to published mean PPD effects through a STRUCTURAL transform):")
for k in PK_ANCHOR_KEYS:
    s = STRAT_OF[k]
    print(f"  {s:14s}: x_label={XLAB[s]:.3f}  closure(m=1)={p_close_mult(s,1.0):.4f}  "
          f"(evidence {RESP[s]:.4f})  T>thr label={TABOVE[k]:.1f} d")
print("  -> rankings now CONSISTENT with the economics (Periochip>Atridox>Arestin); the old "
      "T>MIC-only Emax that made Arestin look best is gone. Mechanism moves closure via m.")

# recurrence survival driven by the SAME unified closure (fixes Fig4D vs Fig5B mismatch)
def recur_survival(t_months, closure, scale_base=18.0, shape=1.3):
    scale = scale_base*(0.5+np.asarray(closure, float))
    return np.exp(-(np.asarray(t_months, float)[..., None]/scale)**shape) if np.ndim(closure) \
           else np.exp(-(np.asarray(t_months, float)/scale)**shape)
grid_m = np.linspace(0, 48, 400)
median_recur = {s: round(float(grid_m[np.argmin(abs(recur_survival(grid_m, RESP[s])-0.5))]), 1) for s in RESP}
print("Median recurrence-free (months), from unified closure:", median_recur)

# In-vitro doxycycline inhibition of neutrophil/GCF collagenase: IC50 15-30 uM.
# We use the conservative 30-uM endpoint for a mechanistic display only; it is NOT mapped to QALY.
DOX_MW = 444.4; IC50_uM = 30.0; IC50_ugml = IC50_uM*DOX_MW/1000
def mmp8_inhib(conc, Imax=1.00, IC50=IC50_ugml, h=1.0):
    c = np.asarray(conc, float); return Imax*c**h/(IC50**h + c**h)
print(f"Doxycycline MMP-8 IC50 = {IC50_ugml:.1f} µg/mL; at tissue 300 µg/mL inhibition = {mmp8_inhib(300)*100:.0f}%")



In [ ]:
# ---- FIGURE 3: mini-PBPK dynamics + 3-product calibration + Sobol GSA (2x2) ----
fig3 = plt.figure(figsize=(13.5, 8.6))
gs = GridSpec(2, 2, figure=fig3, hspace=0.34, wspace=0.26)

axA = fig3.add_subplot(gs[0, 0])
axA.axvspan(0, 14, color=C["periochip"], alpha=0.05)
axA.plot(td, np.clip(Cpf, 1e-2, None), color=C["atridox"], lw=2, label="Pocket fluid (GCF)")
axA.plot(td, np.clip(Cbf, 1e-2, None), color=C["periochip"], lw=2, label="Biofilm")
axA.plot(td, np.clip(Cti, 1e-2, None), color=C["ok"], lw=2, label="Gingival tissue")
axA.axhline(MIC, ls="--", color=C["mic"], lw=1); axA.text(11, MIC*1.15, "MIC", color=C["mic"], fontsize=8)
axA.set_yscale("log"); axA.set_ylim(1, 1100); axA.set_xlabel("Time (days)")
axA.set_ylabel("Concentration (µg/mL, log)"); axA.set_title("Mechanistic 5-compartment mini-PBPK")
axA.legend(loc="upper right", fontsize=7.5); panel_tag(axA, "A")

axB = fig3.add_subplot(gs[0, 1])
for k in PK_ANCHOR_KEYS:
    f = prod_fits[k]; tt2 = np.linspace(0, f["rel"]+1.5, 400)
    axB.plot(tt2, np.clip(pocket_curve_gen(f["p"], f["dose"], f["tmax"], tt2), 1e-2, None),
             color=prod_col[k], lw=1.8, label=f"{k} fit ({f['rtype']})")
    axB.scatter(f["t"], np.clip(f["c"], 1e-2, None), color=prod_col[k], s=20, zorder=5)
    if len(f["t_hold"]):
        axB.scatter(f["t_hold"], np.clip(np.interp(f["t_hold"], f["t"], f["c"]), 1e-2, None),
                    facecolors="none", edgecolors=prod_col[k], s=70, lw=1.4, zorder=6)
axB.set_yscale("log"); axB.set_ylim(1, 3000); axB.set_xlabel("Time (days)")
axB.set_ylabel("GCF concentration (µg/mL, log)")
axB.set_title("Calibration to published summary anchors")
axB.legend(fontsize=6.8, loc="lower left")
axB.text(0.03, 0.97, f"Calibration AAFE: Ato {prod_fits['Atridox']['aafe_full']:.2f}, "
         f"Peri {prod_fits['Periochip']['aafe_full']:.2f}, Ares {prod_fits['Arestin']['aafe_full']:.2f}\n"
         f"Sparse summary anchors: no external holdout claim",
         transform=axB.transAxes, va="top", ha="left", fontsize=6.6,
         bbox=dict(boxstyle="round", fc="white", ec=C["grid"], alpha=0.9))
panel_tag(axB, "B")

def sobol_panel(ax, dfS, title):
    order = dfS.sort_values("ST")["param"].tolist(); yp = np.arange(len(order))
    s1 = dfS.set_index("param").loc[order, "S1"].values; st = dfS.set_index("param").loc[order, "ST"].values
    ax.barh(yp+0.18, s1, 0.34, color=C["cool"], label="First-order S1")
    ax.barh(yp-0.18, st, 0.34, color=C["accent"], label="Total-order ST")
    for j, p in enumerate(order):
        lo1, hi1 = S1ci[0, gsa_names.index(p), 0 if "absorbed" in title.lower() else 1], S1ci[1, gsa_names.index(p), 0 if "absorbed" in title.lower() else 1]
        ax.plot([lo1, hi1], [yp[j]+0.18]*2, color="k", lw=0.7)
    ax.set_yticks(yp); ax.set_yticklabels(order); ax.set_xlabel("Sobol index (95% CI)")
    ax.set_title(title); ax.legend(fontsize=7.5, loc="lower right")
axC = fig3.add_subplot(gs[1, 0]); sobol_panel(axC, sobol_sys, "GSA - drivers of absorbed-sink fraction"); panel_tag(axC, "C")
axD = fig3.add_subplot(gs[1, 1]); sobol_panel(axD, sobol_tmic, "GSA - drivers of T>MIC"); panel_tag(axD, "D")
fig3.suptitle("Figure 3  |  Stage 3: local mini-PBPK - dynamics, anchor calibration and Sobol GSA",
              y=1.01, fontsize=12, fontweight="bold", ha="left", x=0.02)
save_fig(fig3, "Figure03_miniPBPK_calibration_GSA")
print("Saved Figure 3")



In [ ]:
# ---- FIGURE 4: unified closure model + host-modulation + recurrence ----
fig4 = plt.figure(figsize=(14.5, 4.3))
gs = GridSpec(1, 3, figure=fig4, wspace=0.30)

# (A) THE unified exposure->closure curve; products placed by their evidence closure
axA = fig4.add_subplot(gs[0, 0])
xg = np.linspace(0, 2.2, 300)
axA.plot(xg, E0 + (CEIL-E0)*hill(xg), color=C["accent"], lw=2.4, zorder=2)
# uncertainty band from PPD_lo/PPD_hi mapped through closure
for _, row in clinical.iterrows():
    if row["Strategy"] == "SRP alone": continue
    s = row["Strategy"]; xl = XLAB[s]
    axA.scatter([xl], [RESP[s]], s=95, color=prod_col[KEY_OF[s]], zorder=5, edgecolor="white", lw=1)
    axA.annotate(KEY_OF[s], (xl, RESP[s]), textcoords="offset points", xytext=(6, -12), fontsize=8,
                 color=prod_col[KEY_OF[s]])
    clo = closure_prob(row["PPD_lo"]); chi = closure_prob(row["PPD_hi"])
    axA.plot([x_of_closure(clo), x_of_closure(chi)], [clo, chi], color=prod_col[KEY_OF[s]], lw=1.2, alpha=0.5)
axA.scatter([0], [E0], s=70, color=C["srp"], zorder=5); axA.annotate("SRP", (0, E0), xytext=(6, 4),
             textcoords="offset points", fontsize=8, color=C["srp"])
axA.set_xlabel("Relative effective exposure (mechanistic)"); axA.set_ylabel("Pocket closure probability")
axA.set_title("Unified closure model (evidence-anchored)"); axA.set_ylim(0.28, 0.62); panel_tag(axA, "A")
axA.text(0.5, 0.06, "one curve; products placed by meta-analytic\nclosure; PBPK moves them along it",
         transform=axA.transAxes, fontsize=6.8, ha="center", color="#555")

# (B) host modulation: doxycycline anti-MMP-8 vs CHX (none)
axB = fig4.add_subplot(gs[0, 1])
cc = np.logspace(np.log10(0.5), np.log10(500), 200)
axB.plot(cc, mmp8_inhib(cc)*100, color=C["cool"], lw=2.2, label="Doxycycline (anti-MMP-8)")
axB.plot(cc, np.full_like(cc, 3), color=C["periochip"], lw=2.0, ls="--", label="Chlorhexidine (biocide, no anti-MMP)")
axB.axhline(60, ls="--", color=C["mic"], lw=1)
axB.text(0.7, 62, "SDD trial: 60% ↓ MMP-8 odds\n(Caton 2000; Golub)", fontsize=6.8, color="#555")
axB.axvline(IC50_ugml, ls=":", color=C["cool"], lw=1.2)
axB.text(IC50_ugml*1.1, 8, f"IC50 {IC50_ugml:.0f} µg/mL", color=C["cool"], fontsize=7.5)
axB.set_xscale("log"); axB.set_xlabel("Tissue drug concentration (µg/mL, log)")
axB.set_ylabel("MMP-8 inhibition (%)"); axB.set_ylim(0, 90)
axB.set_title("Anti-inflammatory PK/PD (host modulation)"); axB.legend(fontsize=7.5, loc="center left")
panel_tag(axB, "B")

# (C) recurrence survival from the SAME unified closure
axC = fig4.add_subplot(gs[0, 2])
scol = {"SRP alone": C["srp"], "SRP+Atridox": C["atridox"], "SRP+Periochip": C["periochip"], "SRP+Arestin": C["arestin"]}
for s in ["SRP alone", "SRP+Atridox", "SRP+Periochip", "SRP+Arestin"]:
    axC.plot(grid_m, recur_survival(grid_m, RESP[s])*100, color=scol[s], lw=2,
             label=s.replace("SRP alone", "SRP").replace("SRP+", "+"))
axC.axhline(50, ls=":", color=C["mic"], lw=1); axC.set_xlabel("Months since treatment")
axC.set_ylabel("Recurrence-free (%)"); axC.set_title("Recurrence PK/PD (survival)")
axC.legend(fontsize=7.5); panel_tag(axC, "C")
fig4.suptitle("Figure 4  |  Stage 4: coupled local PK/PD - unified closure, host-modulation and recurrence",
              y=1.02, fontsize=12, fontweight="bold", ha="left", x=0.02)
save_fig(fig4, "Figure04_local_PKPD")
print("Saved Figure 4")



In [ ]:
# =====================================================================================
# 5 - Multi-task prediction + PBPK-driven regimen optimization (closure feeds economics)
# =====================================================================================
# Scenario closures now come from the UNIFIED model (product anchor x subgroup host-response
# penalty), so e.g. the diabetic+Arestin scenario correctly reads LOW closure (matching the
# economics), unlike v1 where the T>MIC Emax made Arestin look best.
# Subgroup modifiers below are illustrative stress tests, not literature-derived estimates.
scen = pd.DataFrame([
    ("Non-smoker, shallow, Periochip",   "SRP+Periochip", 1.00, 0.0, "biocide"),
    ("Smoker, deep, Atridox",            "SRP+Atridox",   0.90, 1.0, "antibiotic"),
    ("Diabetic, moderate, Arestin",      "SRP+Arestin",   0.95, 0.0, "antibiotic"),
    ("Smoker+diabetic, deep, Periochip", "SRP+Periochip", 0.85, 1.0, "biocide"),
], columns=["Scenario", "Strategy", "expo_mult", "smoker", "Agent_class"])
def host_penalty(smoker, strat):
    # smokers/diabetics blunt response; antibiotics partially rescue (host modulation)
    pen = 0.90 if smoker else 1.0
    return pen
rows = []
for _, r in scen.iterrows():
    s = r["Strategy"]
    closure = float(p_close_mult(s, r["expo_mult"]) * host_penalty(r["smoker"], s))
    tabove = TABOVE[KEY_OF[s]]*r["expo_mult"]
    med = float(grid_m[np.argmin(abs(recur_survival(grid_m, closure)-0.5))])
    absorbed = prod_fits[KEY_OF[s]]["absorbed_pct"]
    rows.append((r["Scenario"], tabove, closure*100, absorbed, med, r["Agent_class"]))
pred = pd.DataFrame(rows, columns=["Scenario", "Exposure_Tabove_d", "Closure_%", "AbsorbedSink_%", "MedRecur_mo", "Agent_class"])
print("Multi-task scenario prediction (closure from UNIFIED model; Arestin no longer 'best'):")
print(pred.round(1).to_string(index=False))

# ---- regimen optimization: re-dose interval -> 8-week coverage; closure gain fed forward ----
_sd = run_pbpk(PBPK, days=45, n=6000); _sd_t = _sd.t/24
_sd_cbf = _sd.y[2]/PBPK["Vbf"]*1000; _sd_abs = _sd.y[4]
def regimen_metrics(interval_d, n_doses=3, window_d=56):
    tgrid = np.linspace(0, window_d, 5000); dt = tgrid[1]-tgrid[0]
    cbf_tot = np.zeros_like(tgrid); abs_tot = np.zeros_like(tgrid)
    for k in range(n_doses):
        sh = tgrid - k*interval_d
        cbf_tot += np.where(sh >= 0, np.interp(np.clip(sh, 0, None), _sd_t, _sd_cbf), 0.0)
        abs_tot += np.where(sh >= 0, np.interp(np.clip(sh, 0, None), _sd_t, _sd_abs), 0.0)
    days_tmic = float(np.sum(cbf_tot >= CFG["MIC_MINO"])*dt)
    return days_tmic, float(100*abs_tot[-1]/(n_doses*1000.0))
intervals = np.arange(7, 26, 2)
reg = pd.DataFrame([(iv, *regimen_metrics(iv)) for iv in intervals], columns=["Interval_d", "daysTMIC", "absorbedPct"])
best_iv = reg.loc[reg["daysTMIC"].idxmax(), "Interval_d"]
# closure gain from the optimized regimen relative to a single label dose (feeds economics)
single_tmic, _ = regimen_metrics(999, n_doses=1)
best_tmic = reg["daysTMIC"].max()
m_opt = best_tmic/max(single_tmic, 1e-6)
closure_single = p_close_mult("SRP+Arestin", 1.0)
closure_opt = p_close_mult("SRP+Arestin", m_opt)
print(f"\nRegimen optimization (3 doses, 8-wk window): best re-dose interval = {best_iv:.0f} d "
      f"(max T>MIC {best_tmic:.0f} d of 56).")
print(f"  exposure multiplier vs single dose = {m_opt:.2f} -> closure {closure_single:.3f} -> {closure_opt:.3f} "
      f"(mechanism now feeds the economics via the unified curve).")
# reviewer method#4: this re-dosing optimisation is an ILLUSTRATIVE mechanistic exploration only.
# The base-case economics (Tables 4-6, Figures 6-8) use the SINGLE label-dose course for every product;
# the optimised interval does NOT enter the cost-effectiveness or budget-impact results. Adopting a
# 3-dose regimen would require adding its extra drug + chair-time costs to COURSE_COST before re-running.
print(f"  [reviewer note] optimal interval ~{best_iv:.0f} d is illustrative and is NOT used in the "
      f"base-case economics (single label-dose course).")



In [ ]:
# ---- FIGURE 5: multi-task prediction + regimen optimization ----
fig5 = plt.figure(figsize=(14.5, 4.4))
gs = GridSpec(1, 3, figure=fig5, width_ratios=[1.15, 1.0, 1.0], wspace=0.34)

axA = fig5.add_subplot(gs[0, 0])
eff = pred["Closure_%"].values; dur = pred["MedRecur_mo"].values
sink = pred["AbsorbedSink_%"].values
sink_score = 1.0 - sink/np.maximum(sink.max(), 1e-9)  # visualization only; not a safety endpoint
Mtask = np.vstack([eff/eff.max(), dur/dur.max(), sink_score]).T
im = axA.imshow(Mtask, cmap="RdYlGn", vmin=0.6, vmax=1.0, aspect="auto")
axA.set_xticks(range(3)); axA.set_xticklabels(["Efficacy\n(closure)", "Durability\n(recur-free)", "Model sink\n(lower value)"], fontsize=8)
axA.set_yticks(range(len(pred))); axA.set_yticklabels([s[:22] for s in pred["Scenario"]], fontsize=7.5)
for i in range(len(pred)):
    for j, raw in enumerate([eff, dur, sink]):
        axA.text(j, i, f"{[eff,dur,sink][j][i]:.0f}", ha="center", va="center", fontsize=8, fontweight="bold")
axA.set_title("Multi-task prediction (per scenario)"); axA.grid(False); panel_tag(axA, "A")

axB = fig5.add_subplot(gs[0, 1])
sizes = 60 + (sink.max()-sink)*20
labels_short = ["Non-smoker", "Smoker", "Diabetic", "Smoker+diabetic"]
for i in range(len(pred)):
    axB.scatter(eff[i], dur[i], s=200+sizes[i], color=list(prod_col.values())[i%3], alpha=0.7, edgecolor="white")
    axB.annotate(labels_short[i], (eff[i], dur[i]), fontsize=7.5, xytext=(4, 4), textcoords="offset points")
axB.set_xlabel("Predicted pocket closure (%)"); axB.set_ylabel("Median recurrence-free (months)")
axB.set_title("Efficacy-durability view (size=inverse model sink)"); panel_tag(axB, "B")

axC = fig5.add_subplot(gs[0, 2])
axC.plot(reg["Interval_d"], reg["daysTMIC"], "-o", color=C["accent"], lw=2, ms=5)
axC.set_xlabel("Re-dose interval (days)"); axC.set_ylabel("T>MIC (days in 8 wk)", color=C["accent"])
axC.tick_params(axis="y", labelcolor=C["accent"])
axC.axvline(best_iv, ls="--", color=C["ok"], lw=1.5)
axC.text(best_iv+0.3, reg["daysTMIC"].min(), f"optimal\n{best_iv:.0f} d", color=C["ok"], fontsize=8)
ax2 = axC.twinx(); ax2.plot(reg["Interval_d"], reg["absorbedPct"], "--s", color=C["cool"], lw=1.6, ms=4)
ax2.set_ylabel("Absorbed-sink fraction (%)", color=C["cool"]); ax2.tick_params(axis="y", labelcolor=C["cool"])
ax2.grid(False); axC.set_title("Regimen optimization (fixed 3-dose course, 8-wk window)", fontsize=10.5); panel_tag(axC, "C")
fig5.suptitle("Figure 5  |  Stage 5: integrated multi-task prediction and PBPK-driven regimen optimization",
              y=1.02, fontsize=12, fontweight="bold", ha="left", x=0.02)
save_fig(fig5, "Figure05_prediction_regimen")
print("Saved Figure 5")



In [ ]:
# =====================================================================================
# 6 - Individualized optimization + EVIC (precision economics)
#   STRUCTURAL DEMONSTRATION ONLY: costs, utilities, transitions and subgroups are illustrative.
# =====================================================================================
print("WARNING: Economic/subgroup outputs are illustrative structural scenarios, not decision-grade estimates.")

MARKOV_INITIAL_GOOD = np.array([0.70, 0.22, 0.08, 0.0])
MARKOV_INITIAL_POOR = np.array([0.30, 0.40, 0.30, 0.0])
MARKOV_T_GOOD = np.array([[0.965,0.030,0.005,0],[0.120,0.820,0.055,0.005],
                          [0.020,0.150,0.810,0.020],[0,0,0,1.0]])
MARKOV_T_POOR = np.array([[0.920,0.060,0.020,0],[0.070,0.790,0.120,0.020],
                          [0.010,0.080,0.860,0.050],[0,0,0,1.0]])

def markov(p_resp, tx_cost=0.0, recur_mult=1.0, sev_mult=1.0, maint=None, years=None, disc=None, coupling=None):
    maint = CFG["MAINT_COST"] if maint is None else maint
    years = CFG["HORIZON_Y"] if years is None else years
    disc = CFG["DISCOUNT"] if disc is None else disc
    UTIL = CFG["UTIL"]
    coupling = CFG["CLOSURE_MARKOV_COUPLING"] if coupling is None else str(coupling)
    allowed = set(CFG["CLOSURE_MARKOV_COUPLING_SCENARIOS"])
    if coupling not in allowed:
        raise ValueError(f"Unknown closure-Markov coupling: {coupling}")
    p_initial = p_resp if coupling in ("initial_only", "dual") else CFG["P0_SRP"]
    p_transition = p_resp if coupling in ("transition_only", "dual") else CFG["P0_SRP"]
    v = p_initial*MARKOV_INITIAL_GOOD + (1-p_initial)*MARKOV_INITIAL_POOR
    T = p_transition*MARKOV_T_GOOD + (1-p_transition)*MARKOV_T_POOR
    for i in range(3):
        for j in range(i+1, 3):
            ex = T[i, j]*(recur_mult-1); T[i, j] += ex; T[i, i] -= ex
    T = np.clip(T, 0, 1)
    for i in range(4): T[i] /= T[i].sum()
    cs = np.array([maint*0.6, maint, maint*1.6*sev_mult, maint*0.2])
    tc, tq, ty = tx_cost, 0.0, 0.0; dis_cost = 0.0
    for yr in range(years):
        df = 1/(1+disc)**yr
        tc += df*float(v@cs); dis_cost += df*float(v@cs); tq += df*float(v@UTIL); ty += df*float(1-v[3])
        v = v@T
    return dict(cost=tc, qaly=tq, toothyears=ty, disease_cost=dis_cost)

SUBGROUPS = list(CFG["SUBGROUP_PREV"].keys()); STRATS = list(CFG["COURSE_COST"].keys()); WTP = CFG["WTP"]
# ---- reviewer #1 + #2: ONE host-modulation pathway; host modulation demoted to a scenario ----
# v2 double-counted doxycycline host modulation: it entered BOTH as a subgroup closure-rescue
# (raising QALYs through the Markov) AND as a separate additive oral-QALY. v3 splits the two
# channels and activates only ONE (CFG["HM_PATHWAY"]), so no effect is counted twice; strength
# is scaled by an SDD-extrapolation scenario multiplier (CFG["HM_SCENARIO"]).
def hm_mult():
    return CFG["HM_SCEN_MULT"][CFG["HM_SCENARIO"]]
def subgroup_severity(sg):
    """Strategy-INDEPENDENT subgroup effect modifier on closure (smoking / diabetes / depth)."""
    return CFG["SUBGROUP_CLOSURE_DELTA"][sg]
def hm_closure_rescue(strategy, sg):
    """Host modulation acting via improved closure in high-severity subgroups.
    ACTIVE ONLY when HM_PATHWAY=='closure' (else 0.0) -> enforces the single-pathway rule."""
    if CFG["HM_PATHWAY"] != "closure": return 0.0
    r = 0.0
    if sg == "Smoker/deep" and strategy in ("SRP+Atridox", "SRP+Arestin"): r = 0.05
    elif sg == "Diabetic/moderate" and strategy == "SRP+Atridox":          r = 0.05
    elif sg == "Smoker+diabetic" and strategy == "SRP+Atridox":            r = 0.06
    return r * hm_mult()
def closure_delta(strategy, sg):
    """Total closure modifier used everywhere = subgroup severity + (gated) HM closure-rescue."""
    return subgroup_severity(sg) + hm_closure_rescue(strategy, sg)
def _hm_infl(sg):
    return 1.3 if ("iabetic" in sg or "deep" in sg) else 1.0
def hm_utility(strategy, sg, hm_base=None):
    """Host modulation acting via a SEPARATE anti-inflammatory oral-QALY increment.
    ACTIVE ONLY when HM_PATHWAY=='utility' (else 0.0). The MMP-8 -> utility mapping is
    illustrative and scenario-scaled (reviewer #1), anchored to the sub-antimicrobial-dose
    anti-MMP-8 effect (Caton 2000)."""
    if CFG["HM_PATHWAY"] != "utility": return 0.0
    hb = CFG["HM_BASE_QALY"] if hm_base is None else hm_base
    return CFG["HM_FACTOR"][strategy]*hb*_hm_infl(sg)*hm_mult()
# backward-compatible aliases (old call-sites now route through the single-pathway logic)
def subgroup_delta(strategy, sg): return closure_delta(strategy, sg)
def qaly_hostmod(strategy, sg):   return hm_utility(strategy, sg)

recs = []
for sg in SUBGROUPS:
    base = markov(np.clip(RESP["SRP alone"]+subgroup_delta("SRP alone", sg), 0.05, 0.95))
    for strat in STRATS:
        pr = float(np.clip(RESP[strat]+subgroup_delta(strat, sg), 0.05, 0.95))
        m = markov(pr, tx_cost=CFG["COURSE_COST"][strat])
        dQ_am = m["qaly"] - base["qaly"]; dQ_hm = qaly_hostmod(strat, sg); dC = m["cost"] - base["cost"]
        nmb = WTP*(dQ_am + dQ_hm) - dC
        recs.append((sg, strat, pr, dQ_am, dQ_hm, dC, nmb))
opt = pd.DataFrame(recs, columns=["Subgroup","Strategy","p_resp","dQALY_antimic","dQALY_hostmod","dCost","NMB"])
best = opt.loc[opt.groupby("Subgroup")["NMB"].idxmax()][["Subgroup","Strategy","NMB"]]
print(f"Optimal strategy by subgroup (WTP ${WTP:,.0f}/QALY):")
print(best.to_string(index=False))
print(f"Distinct optimal strategies across subgroups: {best['Strategy'].nunique()}")

prev = CFG["SUBGROUP_PREV"]
pop_nmb = (opt.assign(w=opt["Subgroup"].map(prev)).groupby("Strategy")
              .apply(lambda g: np.average(g["NMB"], weights=g["w"]), include_groups=False))
pop_best_strat = pop_nmb.idxmax(); pop_best_val = pop_nmb.max()
indiv_val = sum(prev[sg]*opt[opt["Subgroup"] == sg]["NMB"].max() for sg in SUBGROUPS)
EVIC = indiv_val - pop_best_val

# ---- P1: separate the VALUE OF HOST-MODULATION from the VALUE OF STRATIFICATION ----
# (v1 narratively conflated the Periochip->Atridox "flip" with EVIC.)
opt_noHM = opt.copy(); opt_noHM["NMB_noHM"] = WTP*opt_noHM["dQALY_antimic"] - opt_noHM["dCost"]
pop_nmb_noHM = (opt_noHM.assign(w=opt_noHM["Subgroup"].map(prev)).groupby("Strategy")
                .apply(lambda g: np.average(g["NMB_noHM"], weights=g["w"]), include_groups=False))
pop_best_noHM = pop_nmb_noHM.idxmax(); pop_best_val_noHM = pop_nmb_noHM.max()
value_of_HM = pop_best_val - pop_best_val_noHM     # gain from *including* host modulation (pop-optimal)
print(f"\nPopulation-optimal single strategy: {pop_best_strat} (mean NMB ${pop_best_val:,.0f})")
print(f"Population-optimal WITHOUT host-modulation: {pop_best_noHM} (mean NMB ${pop_best_val_noHM:,.0f})")
print(f"Value of INCLUDING host-modulation (population) = ${value_of_HM:,.0f}/treated site")
print(f"Value of STRATIFICATION (EVIC, within host-mod framework) = ${EVIC:,.0f}/treated site "
      f"-> ${EVIC*1e4:,.0f} per 10,000. (Reported SEPARATELY, per P1.)")

# ---- reviewer #1: host modulation is an SDD-extrapolated SCENARIO, not an established base case ----
# Re-derive the prevalence-weighted population-optimal strategy under each pathway x strength so the
# fragility of the doxycycline conclusion to the host-modulation assumption is explicit.
def _pop_optimal(pathway, scenario):
    keep_p, keep_s = CFG["HM_PATHWAY"], CFG["HM_SCENARIO"]
    CFG["HM_PATHWAY"], CFG["HM_SCENARIO"] = pathway, scenario
    rows = []
    for sg in SUBGROUPS:
        b = markov(np.clip(RESP["SRP alone"]+closure_delta("SRP alone", sg), 0.05, 0.95))
        for st in STRATS:
            pr = float(np.clip(RESP[st]+closure_delta(st, sg), 0.05, 0.95))
            mm = markov(pr, tx_cost=CFG["COURSE_COST"][st])
            nb = WTP*((mm["qaly"]-b["qaly"]) + hm_utility(st, sg)) - (mm["cost"]-b["cost"])
            rows.append((sg, st, nb))
    dfp = pd.DataFrame(rows, columns=["sg","st","nb"])
    pn = (dfp.assign(w=dfp["sg"].map(prev)).groupby("st")
             .apply(lambda g: np.average(g["nb"], weights=g["w"]), include_groups=False))
    CFG["HM_PATHWAY"], CFG["HM_SCENARIO"] = keep_p, keep_s
    return pn.idxmax(), float(pn.max())
print("\nreviewer #1 - population-optimal strategy vs host-modulation assumption (pathway x strength):")
for pw in ("none", "closure", "utility"):
    scens = ("current",) if pw == "none" else ("weaker", "current")
    line = []
    for sc in scens:
        who, val = _pop_optimal(pw, sc)
        line.append(f"{sc}:{who.replace('SRP+','+')} (${val:,.0f})")
    print(f"  HM pathway={pw:8s}: " + " | ".join(line))
print("  -> the population-optimal at the deterministic point estimate is shown above for each pathway x "
      "strength; once the double count is removed, any doxycycline advantage is confined to a MINORITY of "
      "PSA draws (see P(optimal) below) rather than the point estimate. It is therefore uncertainty/"
      "assumption-driven and must be reported as such, not as an established base-case recommendation.")

host_modulation_sensitivity_rows = []
for pathway, scenario in [("none", "none"), ("closure", "weaker"), ("closure", "current"),
                          ("utility", "weaker"), ("utility", "current")]:
    winner, winner_nmb = _pop_optimal(pathway, scenario)
    host_modulation_sensitivity_rows.append(
        (pathway, scenario, CFG["HM_SCEN_MULT"][scenario], winner, winner_nmb,
         "base case excludes MMP-8 QALY" if pathway == "none" else "exploratory single-pathway scenario")
    )
host_modulation_sensitivity = pd.DataFrame(
    host_modulation_sensitivity_rows,
    columns=["Host-modulation pathway", "Strength scenario", "Strength multiplier",
             "Population-optimal strategy", "Illustrative NMB", "Interpretation"],
)
print("\nHost-modulation exclusion and single-pathway sensitivity:")
print(host_modulation_sensitivity.to_string(index=False))




In [ ]:
# ---- FIGURE 7: individualized optimization + QALY decomposition + EVIC ----
fig7 = plt.figure(figsize=(15.5, 4.5))
gs = GridSpec(1, 3, figure=fig7, width_ratios=[1.15, 1.0, 1.0], wspace=0.36)

axA = fig7.add_subplot(gs[0, 0])
piv = opt.pivot(index="Subgroup", columns="Strategy", values="NMB").reindex(index=SUBGROUPS, columns=STRATS)
im = axA.imshow(piv.values/1000, cmap="RdYlGn", aspect="auto")
best_col = {sg: opt[opt["Subgroup"] == sg].set_index("Strategy")["NMB"].idxmax() for sg in SUBGROUPS}
for i, sg in enumerate(SUBGROUPS):
    for j, st in enumerate(STRATS):
        val = piv.loc[sg, st]/1000; star = "★" if best_col[sg] == st else ""
        axA.text(j, i, f"{val:.1f}k{star}", ha="center", va="center", fontsize=8,
                 fontweight="bold" if star else "normal")
axA.set_xticks(range(len(STRATS))); axA.set_xticklabels([s.replace("SRP alone","SRP").replace("SRP+","+") for s in STRATS], rotation=25, ha="right", fontsize=8)
axA.set_yticks(range(len(SUBGROUPS))); axA.set_yticklabels(SUBGROUPS, fontsize=8)
axA.set_title("Individual NMB (★ optimal per subgroup)"); axA.grid(False); panel_tag(axA, "A")

axB = fig7.add_subplot(gs[0, 1])
dec = opt.groupby("Strategy")[["dQALY_antimic", "dQALY_hostmod"]].mean().reindex(["SRP+Atridox","SRP+Periochip","SRP+Arestin"])
xp = np.arange(len(dec))
axB.bar(xp, dec["dQALY_antimic"], color=C["srp"], label="Antimicrobial QALY")
axB.bar(xp, dec["dQALY_hostmod"], bottom=dec["dQALY_antimic"], color=C["cool"], label="Host-modulation QALY")
for i, st in enumerate(dec.index):
    axB.text(i, dec["dQALY_antimic"].iloc[i]/2, f"{dec['dQALY_antimic'].iloc[i]:.3f}", ha="center", color="white", fontsize=7.5)
    if dec["dQALY_hostmod"].iloc[i] > 0.001:
        axB.text(i, dec["dQALY_antimic"].iloc[i]+dec["dQALY_hostmod"].iloc[i]+0.004, f"+{dec['dQALY_hostmod'].iloc[i]:.3f}",
                 ha="center", color=C["cool"], fontsize=7.5, fontweight="bold")
axB.set_xticks(xp); axB.set_xticklabels([s.replace("SRP+","") for s in dec.index])
axB.set_ylabel("Mean incremental oral-QALY"); axB.set_title("QALY decomposition (Caton 2000 anchor)")
axB.legend(fontsize=7.5); panel_tag(axB, "B")

axC = fig7.add_subplot(gs[0, 2])
indiv_by_sg = [opt[opt["Subgroup"] == sg]["NMB"].max() for sg in SUBGROUPS]
pop_by_sg = [opt[(opt["Subgroup"] == sg) & (opt["Strategy"] == pop_best_strat)]["NMB"].values[0] for sg in SUBGROUPS]
yp = np.arange(len(SUBGROUPS))
axC.barh(yp+0.2, indiv_by_sg, 0.4, color=C["ok"], label="Individualized (best/subgroup)")
axC.barh(yp-0.2, pop_by_sg, 0.4, color=C["srp"], label=f"Population ({pop_best_strat.replace('SRP+','+')})")
axC.set_yticks(yp); axC.set_yticklabels(SUBGROUPS, fontsize=8); axC.set_xlabel("NMB ($/treated site)")
axC.set_title(f"Value of stratification: EVIC=${EVIC:,.0f}/site"); axC.legend(fontsize=7.5, loc="lower right")
panel_tag(axC, "C")
fig7.suptitle("Figure 7  |  Stage 7: individualized optimization and the value of stratification (precision economics)",
              y=1.02, fontsize=12, fontweight="bold", ha="left", x=0.02)
save_fig(fig7, "Figure07_individualized_EVIC")
print("Saved Figure 7")



In [ ]:
# =====================================================================================
# 7a - Base-case CEA + frontier, and P0-2: BIA downstream savings DERIVED from the Markov
# =====================================================================================
arms = {s: markov(RESP[s], tx_cost=CFG["COURSE_COST"][s]) for s in STRATS}
base = arms["SRP alone"]
ce = pd.DataFrame([(s, arms[s]["cost"], arms[s]["qaly"], arms[s]["toothyears"],
                    arms[s]["cost"]-base["cost"], arms[s]["qaly"]-base["qaly"],
                    arms[s]["toothyears"]-base["toothyears"]) for s in STRATS],
                  columns=["Arm","Cost","QALY","ToothYrs","dCost","dQALY","dToothYrs"])
ce["ICER_vs_SRP"] = np.where(ce["dQALY"].abs() > 1e-9, ce["dCost"]/ce["dQALY"], np.nan)

def cea_frontier(df):
    d = df.sort_values("Cost").reset_index(drop=True); eff = []
    for i in range(len(d)):
        dominated = ((d["Cost"] <= d.loc[i,"Cost"]) & (d["QALY"] >= d.loc[i,"QALY"]) &
                     ((d["Cost"] < d.loc[i,"Cost"]) | (d["QALY"] > d.loc[i,"QALY"]))).any()
        if not dominated: eff.append(i)
    frontier = d.loc[eff].sort_values("QALY").reset_index(drop=True); changed = True
    while changed and len(frontier) > 2:
        changed = False; ic = [np.nan]
        for j in range(1, len(frontier)):
            dc = frontier.loc[j,"Cost"]-frontier.loc[j-1,"Cost"]; dq = frontier.loc[j,"QALY"]-frontier.loc[j-1,"QALY"]
            ic.append(dc/dq if dq > 0 else np.inf)
        for j in range(2, len(frontier)):
            if ic[j] < ic[j-1]:
                frontier = frontier.drop(index=j-1).reset_index(drop=True); changed = True; break
    return frontier
frontier = cea_frontier(ce)
seq = frontier.copy()
seq["seqICER"] = [np.nan] + [(seq.loc[j,"Cost"]-seq.loc[j-1,"Cost"])/(seq.loc[j,"QALY"]-seq.loc[j-1,"QALY"]) for j in range(1, len(seq))]
print("Base-case cost-effectiveness (payer, 20 y, 3% discount):")
print(ce.round(3).to_string(index=False))
print("\nEfficient frontier (dominated/extended-dominated removed), sequential ICER:")
print(seq[["Arm","Cost","QALY","seqICER"]].round(3).to_string(index=False))

# ---- reviewer #3: classify each OFF-frontier adjunct as strongly (simple) vs extended dominated ----
_front_arms = set(frontier["Arm"])
for _, r in ce.iterrows():
    if r["Arm"] in _front_arms or r["Arm"] == "SRP alone":
        continue
    _strong = ((ce["Cost"] <= r["Cost"]) & (ce["QALY"] >= r["QALY"]) &
               ((ce["Cost"] < r["Cost"]) | (ce["QALY"] > r["QALY"]))).any()
    print(f"  dominance: {r['Arm']:14s} -> {'STRONGLY (simple) dominated' if _strong else 'extended-dominated'} "
          f"(cost ${r['Cost']:.0f}, QALY {r['QALY']:.3f})")

# ---- reviewer #10: analysis-unit statement + external plausibility of the QALY gain ----
print(f"\nAnalysis unit: one representative treated periodontal site (index tooth); per-treated-site results "
      f"assume N_INDEX_SITES={CFG['N_INDEX_SITES']} treated site(s) per course.")
for s in STRATS:
    if s == "SRP alone": continue
    dQ = arms[s]["qaly"] - base["qaly"]; days = dQ*365.25
    flag = "  <-- CHECK: large for a single local adjunct" if abs(days) > 60 else ""
    print(f"  plausibility: {s:14s} dQALY vs SRP = {dQ:+.3f}  (~{days:+.0f} healthy-day equivalents){flag}")

# ---- P0-2 FIX: downstream saving per adjunct = reduction in discounted DISEASE cost vs SRP
DOWNSTREAM_SAVE = {s: base["disease_cost"] - arms[s]["disease_cost"] for s in STRATS}
print("\nP0-2 - downstream saving DERIVED from the Markov disease-cost stream (not hard-coded):")
for s in STRATS:
    if s == "SRP alone": continue
    net = CFG["COURSE_COST"][s] - DOWNSTREAM_SAVE[s]
    print(f"  {s:14s}: drug ${CFG['COURSE_COST'][s]:.0f} - downstream saving ${DOWNSTREAM_SAVE[s]:.0f} "
          f"= NET ${net:+.0f}/site  ({'cost-incurring' if net>0 else 'cost-SAVING'})  [v1 hard-coded $520 saving]")

# ---- Reviewer-requested alternative PPD-to-closure mapping sensitivity ----
def cea_under_mapping(mapping, chx_effect, coupling=None):
    wmd_values = {row.Strategy: float(row.PPD_WMD) for row in clinical.itertuples()}
    wmd_values["SRP+Periochip"] = float(chx_effect)
    closure_values = {strategy: closure_prob(wmd, mapping=mapping)
                      for strategy, wmd in wmd_values.items()}
    coupling = CFG["CLOSURE_MARKOV_COUPLING"] if coupling is None else coupling
    outcomes = {strategy: markov(closure_values[strategy], tx_cost=CFG["COURSE_COST"][strategy], coupling=coupling)
                for strategy in STRATS}
    frame = pd.DataFrame([
        (strategy, outcomes[strategy]["cost"], outcomes[strategy]["qaly"], closure_values[strategy])
        for strategy in STRATS
    ], columns=["Arm", "Cost", "QALY", "Closure"])
    frame["NMB"] = CFG["WTP"] * frame["QALY"] - frame["Cost"]
    winner = frame.loc[frame["NMB"].idxmax(), "Arm"]
    srp = frame.loc[frame["Arm"].eq("SRP alone")].iloc[0]
    chip = frame.loc[frame["Arm"].eq("SRP+Periochip")].iloc[0]
    gel = frame.loc[frame["Arm"].eq("SRP+Atridox")].iloc[0]
    chip_icer = ((chip["Cost"] - srp["Cost"]) / (chip["QALY"] - srp["QALY"])
                 if chip["QALY"] > srp["QALY"] else np.nan)
    return {
        "Mapping": mapping,
        "Markov coupling": coupling,
        "CHX PPD effect (mm)": chx_effect,
        "SRP closure": closure_values["SRP alone"],
        "Atridox closure": closure_values["SRP+Atridox"],
        "PerioChip closure": closure_values["SRP+Periochip"],
        "Arestin closure": closure_values["SRP+Arestin"],
        "Optimal strategy at illustrative WTP": winner,
        "INMB PerioChip vs Atridox": chip["NMB"] - gel["NMB"],
        "PerioChip ICER vs SRP": chip_icer,
    }


mapping_sensitivity = pd.DataFrame([
    cea_under_mapping(mapping, chx_effect)
    for mapping in CFG["CLOSURE_MAPPINGS"]
    for chx_effect in (0.30, 0.75)
])
print("\nAlternative PPD-to-closure mapping sensitivity (proof-of-concept economics):")
print(mapping_sensitivity.round(3).to_string(index=False))
print("  -> Monotone mappings preserve the point-estimate PPD ordering by construction; the low CHX")
print("     evidence scenario tests whether the decision ranking reverses when the clinical input changes.")

# The base case now passes closure into the Markov model once, through the post-treatment
# initial-state distribution.  These scenarios show how results change if closure instead
# affects transitions, or is allowed to affect both components as in the prior implementation.
closure_coupling_sensitivity = pd.DataFrame([
    cea_under_mapping(CFG["CLOSURE_MAPPING"], chx_effect, coupling=coupling)
    for coupling in CFG["CLOSURE_MARKOV_COUPLING_SCENARIOS"]
    for chx_effect in (0.30, 0.75)
])
print("\nClosure-to-Markov coupling sensitivity (base mapping):")
print(closure_coupling_sensitivity.round(3).to_string(index=False))
print("  -> initial_only is the conservative base case; dual is retained only as a sensitivity scenario.")

figS1, (ax_map, ax_inmb) = plt.subplots(1, 2, figsize=(11.5, 4.2))
ppd_grid = np.linspace(0, 1.10, 150)
for mapping in CFG["CLOSURE_MAPPINGS"]:
    ax_map.plot(ppd_grid, [closure_prob(value, mapping=mapping) for value in ppd_grid],
                lw=2, label=mapping)
ax_map.set_xlabel("Incremental PPD WMD (mm)")
ax_map.set_ylabel("Structural pocket-closure proxy")
ax_map.set_title("Alternative structural mappings")
ax_map.legend()

chx_grid = np.linspace(0.30, 0.75, 60)
for mapping in CFG["CLOSURE_MAPPINGS"]:
    inmb = [cea_under_mapping(mapping, value)["INMB PerioChip vs Atridox"] for value in chx_grid]
    ax_inmb.plot(chx_grid, np.asarray(inmb) / 1000, lw=2, label=mapping)
ax_inmb.axhline(0, color="black", lw=1, ls="--")
ax_inmb.set_xlabel("Chlorhexidine PPD effect (mm)")
ax_inmb.set_ylabel("INMB: PerioChip - Atridox ($000/site)")
ax_inmb.set_title("Decision sensitivity across mappings")
ax_inmb.legend()
figS1.suptitle("Supplementary Figure S1 | PPD-to-closure structural sensitivity")
figS1.tight_layout()
save_fig(figS1, "FigureS01_closure_mapping_sensitivity")
print("Saved Supplementary Figure S1")




In [ ]:
# =====================================================================================
# 7b - Budget-impact analysis: TWO scenarios, both consistent with the Markov (P0-2)
# -------------------------------------------------------------------------------------
# v1 hard-coded a $520/site downstream saving -> a -$2.5M "cost-saving" that CONTRADICTED the
# CEA (where every adjunct is net cost-additive). Here the downstream saving is READ from the
# same Markov disease-cost stream, so BIA and CEA agree by construction. We show:
#   (A) base case  : Markov-derived saving -> cost-INCURRING (the correct sign);
#   (B) break-even : the saving that would be needed to reach budget neutrality (and how far
#                    v1's $520 assumption sits beyond it).
POP_TREATED, BIA_LABEL = 10000, "10,000 treated sites (lifetime, discounted)"
POP_BEST = pop_best_strat  # population-optimal strategy incl. host-modulation (from Stage 7)
def bia_scenarios(strat):
    course = CFG["COURSE_COST"][strat]
    save_markov = DOWNSTREAM_SAVE[strat]              # discounted lifetime disease-cost reduction
    net_base = course - save_markov                   # >0 => cost-incurring
    save_breakeven = course                           # saving needed for net = 0
    net_v1 = course - 520.0                            # v1's hard-coded assumption
    return dict(course=course, save_markov=save_markov, net_base=net_base,
                save_breakeven=save_breakeven, net_v1=net_v1)
bia = {s: bia_scenarios(s) for s in STRATS if s != "SRP alone"}
print("Budget-impact analysis -", BIA_LABEL)
print(f"  (downstream saving now DERIVED from the Markov, so BIA agrees with the CEA in sign)")
for s, b in bia.items():
    print(f"  {s:14s}: course ${b['course']:.0f} | Markov saving ${b['save_markov']:.0f} "
          f"-> NET ${b['net_base']:+.0f}/site  => 5-scale budget {POP_TREATED*b['net_base']/1e6:+.2f}M "
          f"(cost-{'incurring' if b['net_base']>0 else 'saving'})")
print(f"  Break-even saving to reach neutrality = each arm's course cost "
      f"(Periochip needs ${bia['SRP+Periochip']['save_breakeven']:.0f} vs Markov ${bia['SRP+Periochip']['save_markov']:.0f}); "
      f"v1's $520 assumption implies a NET ${bia['SRP+Periochip']['net_v1']:+.0f}/site, i.e. it overstated savings and flipped the sign.")



In [ ]:
# =====================================================================================
# 7c - ONE shared probabilistic sample (PSA) feeding CEAC, cross-fitted EVPPI, P(optimal),
#      the decision tornado and the winner-flip curves. (P0-3 + P1)
# -------------------------------------------------------------------------------------
# Population host-modulation inflation = prevalence-weighted subgroup factor (matches Stage 7).
INFL_AVG = float(sum(CFG["SUBGROUP_PREV"][sg]*(1.3 if ("iabetic" in sg or "deep" in sg) else 1.0)
                     for sg in SUBGROUPS))
def hm_pop(strat, hm_base):
    # homogeneous-lens host-modulation utility (population-average inflation); gated + scenario-scaled
    if CFG["HM_PATHWAY"] != "utility": return 0.0
    return CFG["HM_FACTOR"][strat]*hm_base*INFL_AVG*hm_mult()

# meta-analytic SD of each adjunct's PPD WMD from its published 95% CI (clinical table)
WMD_MEAN = {r.Strategy: r.PPD_WMD for r in clinical.itertuples() if r.Strategy != "SRP alone"}
WMD_SD = {r.Strategy: max((r.PPD_hi - r.PPD_lo)/(2*1.96), 1e-3)
          for r in clinical.itertuples() if r.Strategy != "SRP alone"}
RHO_WMD = 0.0    # no empirical joint covariance was available; do not invent correlation

def arm_outcome(strat, wmd, hm_base, cost_mult, recur, wtp):
    closure = closure_prob(wmd) if strat != "SRP alone" else RESP["SRP alone"]
    m = markov(closure, tx_cost=CFG["COURSE_COST"][strat]*cost_mult,
               recur_mult=recur, maint=CFG["MAINT_COST"]*cost_mult)
    qaly = m["qaly"] + hm_pop(strat, hm_base)
    nb = wtp*qaly - m["cost"]
    return nb, m["cost"], qaly, m["disease_cost"]

N_PSA = 5000
z0 = rng.standard_normal(N_PSA)
wmd_draw = {}
for s in WMD_MEAN:
    if s == "SRP+Periochip":
        # Evidence-scenario distribution between the FDA pivotal ~0.30-mm increment at 9 months
        # and the Ma-Diao 0.75-mm meta-analytic estimate at 6 months; NOT a statistical CI.
        wmd_draw[s] = rng.uniform(0.30, 0.75, N_PSA)
    else:
        zi = rng.standard_normal(N_PSA)
        wmd_draw[s] = np.clip(WMD_MEAN[s] + WMD_SD[s]*zi, 0.0, None)
hm_draw = np.zeros(N_PSA)  # no empirical local-LDD-to-QALY mapping; disabled in evidence-audited base
cost_draw = rng.gamma(1/0.15**2, 0.15**2, N_PSA)          # mean 1, CV 15%
recur_draw = np.clip(rng.normal(1.0, 0.12, N_PSA), 0.7, 1.6)
# ---- reviewer #6 / structural-uncertainty: sample the closure-MAPPING parameters ----
def _trunc_normal(mean, sd, lo, hi, n):
    return np.clip(rng.normal(mean, sd, n), lo, hi)
if CFG["PROPAGATE_STRUCTURAL_PSA"]:
    _sp = CFG["STRUCT_PSA"]
    p0_draw    = _trunc_normal(*_sp["P0_SRP"], N_PSA)
    kappa_draw = _trunc_normal(*_sp["KAPPA_CLOSE"], N_PSA)
    ceil_draw  = _trunc_normal(*_sp["PCLOSE_CEIL"], N_PSA)
else:
    p0_draw    = np.full(N_PSA, CFG["P0_SRP"])
    kappa_draw = np.full(N_PSA, CFG["KAPPA_CLOSE"])
    ceil_draw  = np.full(N_PSA, CFG["PCLOSE_CEIL"])
WTP0 = CFG["WTP"]

# ------------------------------------------------------------------------------------
# SUBGROUP-PREVALENCE-WEIGHTED population decision (same lens as Stage 7): each draw is
# scored over the 4 subgroups (subgroup closure heterogeneity via subgroup_delta +
# host-modulation), then prevalence-weighted. This makes P(optimal)/CEAC/EVPPI consistent
# with the Stage-6 population-optimal strategy. (The Fig-7A/B CE plane & BIA remain the
# homogeneous "representative treated-site reference case"; the two lenses are labelled as such.)
prev_arr = np.array([prev[sg] for sg in SUBGROUPS])
def _infl(sg): return 1.3 if ("iabetic" in sg or "deep" in sg) else 1.0
def psa_draw(wmd_d, hm_b, cm, rc, st_p0, st_kap, st_ceil, mapping=None):
    Cm = np.zeros((len(SUBGROUPS), len(STRATS))); Qm = np.zeros_like(Cm)
    for i, sg in enumerate(SUBGROUPS):
        for j, s in enumerate(STRATS):
            w = 0.0 if s == "SRP alone" else wmd_d[s]
            closure = float(np.clip(closure_prob(w, p0=st_p0, kappa=st_kap, ceil=st_ceil, mapping=mapping)
                                    + closure_delta(s, sg), 0.05, 0.95))
            m = markov(closure, tx_cost=CFG["COURSE_COST"][s]*cm, recur_mult=rc, maint=CFG["MAINT_COST"]*cm)
            Cm[i, j] = m["cost"]; Qm[i, j] = m["qaly"] + hm_utility(s, sg, hm_base=hm_b)
    return prev_arr @ Cm, prev_arr @ Qm, (WTP0*Qm - Cm)      # Cpop, Qpop, per-subgroup NB

Cpop_all = np.zeros((N_PSA, len(STRATS))); Qpop_all = np.zeros((N_PSA, len(STRATS)))
evic_draws = np.zeros(N_PSA); strat_matters = np.zeros(N_PSA, bool)
for d in range(N_PSA):
    wmd_d = {s: wmd_draw[s][d] for s in WMD_MEAN}
    Cpop, Qpop, nmb_sg = psa_draw(wmd_d, hm_draw[d], cost_draw[d], recur_draw[d],
                                  p0_draw[d], kappa_draw[d], ceil_draw[d])
    Cpop_all[d] = Cpop; Qpop_all[d] = Qpop
    NBpop = WTP0*Qpop - Cpop; pop_arg = int(NBpop.argmax())
    evic_draws[d] = float(prev_arr @ nmb_sg.max(1)) - float(NBpop.max())
    strat_matters[d] = bool(np.any(nmb_sg.argmax(1) != pop_arg))   # FP-robust "EVIC>0" indicator
evic_draws = np.clip(evic_draws, 0.0, None)   # EVIC is theoretically >=0; remove tiny FP negatives

# Reference-case PSA used for the uncertainty intervals reported in main Table 2.
# This deliberately matches the homogeneous, representative treated-site lens of the
# deterministic Table 2 rather than the subgroup-weighted population-decision lens above.
Cref_all = np.zeros((N_PSA, len(STRATS))); Qref_all = np.zeros_like(Cref_all)
for d in range(N_PSA):
    for j, s in enumerate(STRATS):
        w = 0.0 if s == "SRP alone" else wmd_draw[s][d]
        closure = closure_prob(w, p0=p0_draw[d], kappa=kappa_draw[d], ceil=ceil_draw[d])
        outcome = markov(
            closure,
            tx_cost=CFG["COURSE_COST"][s] * cost_draw[d],
            recur_mult=recur_draw[d],
            maint=CFG["MAINT_COST"] * cost_draw[d],
        )
        Cref_all[d, j] = outcome["cost"]
        Qref_all[d, j] = outcome["qaly"]

reference_index = STRATS.index("SRP alone")
reference_case_psa_icer_rows = []
for j, s in enumerate(STRATS):
    if j == reference_index:
        continue
    delta_cost = Cref_all[:, j] - Cref_all[:, reference_index]
    delta_qaly = Qref_all[:, j] - Qref_all[:, reference_index]
    positive_qaly = np.isfinite(delta_cost) & np.isfinite(delta_qaly) & (delta_qaly > 1e-12)
    icer = delta_cost[positive_qaly] / delta_qaly[positive_qaly]
    dc_lo, dc_med, dc_hi = np.percentile(delta_cost, [2.5, 50, 97.5])
    dq_lo, dq_med, dq_hi = np.percentile(delta_qaly, [2.5, 50, 97.5])
    ic_lo, ic_med, ic_hi = np.percentile(icer, [2.5, 50, 97.5])
    reference_case_psa_icer_rows.append({
        "Arm": s,
        "PSA draws": N_PSA,
        "Draws with positive dQALY": int(positive_qaly.sum()),
        "P(dQALY <= 0)": float(np.mean(delta_qaly <= 0)),
        "dCost 2.5th": float(dc_lo), "dCost median": float(dc_med), "dCost 97.5th": float(dc_hi),
        "dQALY 2.5th": float(dq_lo), "dQALY median": float(dq_med), "dQALY 97.5th": float(dq_hi),
        "ICER 2.5th (positive dQALY draws)": float(ic_lo),
        "ICER median (positive dQALY draws)": float(ic_med),
        "ICER 97.5th (positive dQALY draws)": float(ic_hi),
    })
reference_case_psa_icer_ranges = pd.DataFrame(reference_case_psa_icer_rows)
print("Reference-case PSA ICER percentiles vs SRP (conditional on positive incremental QALYs):")
print(reference_case_psa_icer_ranges[[
    "Arm", "P(dQALY <= 0)", "ICER 2.5th (positive dQALY draws)",
    "ICER median (positive dQALY draws)", "ICER 97.5th (positive dQALY draws)",
]].round(3).to_string(index=False))

# PSA table (prevalence-weighted population cost/QALY/NB per arm + sampled parameters)
psa = pd.DataFrame({"hm_base": hm_draw, "cost_mult": cost_draw, "recur": recur_draw,
                    "p0": p0_draw, "kappa": kappa_draw, "ceil": ceil_draw})
for s in WMD_MEAN: psa["wmd_"+KEY_OF[s]] = wmd_draw[s]
NB0 = WTP0*Qpop_all - Cpop_all
for j, s in enumerate(STRATS):
    psa["C_"+s] = Cpop_all[:, j]; psa["Q_"+s] = Qpop_all[:, j]; psa["NB_"+s] = NB0[:, j]
NB_COLS = ["NB_"+s for s in STRATS]
psa_best = NB0.argmax(1)
p_optimal = pd.Series(np.bincount(psa_best, minlength=len(STRATS))/N_PSA, index=STRATS)
print("P(each strategy is optimal) at WTP ${:,.0f}/QALY  [subgroup-weighted PSA, N={}]:".format(WTP0, N_PSA))
print(p_optimal.round(3).to_string())
print("  (consistent with the Stage-6 population-optimal strategy: {}).".format(pop_best_strat))
print("Population-mean NB by arm:", {s: round(float(NB0[:, j].mean()),0) for j, s in enumerate(STRATS)})

# ---- CEAC across WTP (NB linear in WTP: re-score from stored prevalence-weighted C & Q) ----
WTP_GRID = np.linspace(0, 100000, 41)
ceac = np.zeros((len(WTP_GRID), len(STRATS)))
for wi, w in enumerate(WTP_GRID):
    win = (w*Qpop_all - Cpop_all).argmax(1); ceac[wi] = np.bincount(win, minlength=len(STRATS))/N_PSA
ceac_df = pd.DataFrame(ceac, columns=STRATS, index=np.round(WTP_GRID/1000,0))

# ---- EVPI and cross-fitted EVPPI metamodels ----
EVPI = max(float(NB0.max(1).mean() - NB0.mean(0).max()), 0.0)

def evppi_group(param_cols):
    """Five-fold out-of-sample EVPPI estimate with an explicit theoretical bound."""
    Xg = psa[param_cols].values
    if np.all(np.ptp(Xg, axis=0) <= np.finfo(float).eps):
        return 0.0, 0.0
    cond = np.zeros((N_PSA, len(STRATS)))
    folds = KFold(n_splits=5, shuffle=True, random_state=RNG_SEED + 1701)
    for train_index, test_index in folds.split(Xg):
        for strategy_index, strategy in enumerate(STRATS):
            gam = make_pipeline(
                SplineTransformer(degree=3, n_knots=6, include_bias=False),
                LinearRegression(),
            )
            gam.fit(Xg[train_index], NB0[train_index, strategy_index])
            cond[test_index, strategy_index] = gam.predict(Xg[test_index])
    raw_cross_fitted = max(
        float(cond.max(1).mean() - NB0.mean(0).max()),
        0.0,
    )
    # Sampling and metamodel error can produce a slight finite-sample violation.
    # Retain that diagnostic, but constrain the reported decision-theoretic quantity.
    return raw_cross_fitted, min(raw_cross_fitted, EVPI)

EVPPI_GROUPS = {
    "Closure/efficacy (PPD WMD)": ["wmd_Atridox","wmd_Periochip","wmd_Arestin"],
    "Host-modulation QALY (disabled)": ["hm_base"],
    "Recurrence/retention":       ["recur"],
    "Costs":                      ["cost_mult"],
}
if CFG["PROPAGATE_STRUCTURAL_PSA"]:
    EVPPI_GROUPS["Closure mapping (P0/kappa/ceiling)"] = ["p0", "kappa", "ceil"]
evppi_fit = {name: evppi_group(columns) for name, columns in EVPPI_GROUPS.items()}
evppi_raw = {name: values[0] for name, values in evppi_fit.items()}
evppi = {name: values[1] for name, values in evppi_fit.items()}
print(f"\nEVPI (overall) = ${EVPI:,.0f}/treated site  -> ${EVPI*POP_TREATED:,.0f} per {POP_TREATED:,}")
print("Five-fold cross-fitted EVPPI by parameter group (out-of-sample GAM predictions):")
print("Reported EVPPI is constrained to [0, EVPI]; unconstrained estimates are retained for diagnostics.")
for name, value in sorted(evppi.items(), key=lambda item: -item[1]):
    print(f"  {name:35s}: ${value:,.0f}/site  ({100*value/max(EVPI,1e-9):.0f}% of EVPI; raw ${evppi_raw[name]:,.1f})")

# ---- Reviewer-requested same-draw PSA under alternative closure mappings ----
mapping_nb_arrays = {CFG["CLOSURE_MAPPING"]: NB0.copy()}
for mapping in CFG["CLOSURE_MAPPINGS"]:
    if mapping == CFG["CLOSURE_MAPPING"]:
        continue
    nb_mapping = np.zeros_like(NB0)
    for draw_index in range(N_PSA):
        wmd_values = {strategy: wmd_draw[strategy][draw_index] for strategy in WMD_MEAN}
        c_map, q_map, _ = psa_draw(
            wmd_values, hm_draw[draw_index], cost_draw[draw_index], recur_draw[draw_index],
            p0_draw[draw_index], kappa_draw[draw_index], ceil_draw[draw_index], mapping=mapping,
        )
        nb_mapping[draw_index] = WTP0 * q_map - c_map
    mapping_nb_arrays[mapping] = nb_mapping

mapping_psa_rows = []
for mapping, nb_mapping in mapping_nb_arrays.items():
    winners = nb_mapping.argmax(axis=1)
    probabilities = np.bincount(winners, minlength=len(STRATS)) / N_PSA
    for strategy_index, strategy in enumerate(STRATS):
        probability = float(probabilities[strategy_index])
        mapping_psa_rows.append(
            (mapping, strategy, probability, np.sqrt(probability * (1 - probability) / N_PSA),
             float(nb_mapping[:, strategy_index].mean()))
        )

scenario_mixture_nb = np.stack([mapping_nb_arrays[mapping] for mapping in CFG["CLOSURE_MAPPINGS"]], axis=0)
scenario_mixture_winners = scenario_mixture_nb.argmax(axis=2)
for strategy_index, strategy in enumerate(STRATS):
    # Each PSA draw is a cluster shared across the four mappings.  Averaging the four
    # within-draw winner indicators and estimating the SE over 5,000 draw clusters avoids
    # falsely treating the 20,000 correlated mapping/draw combinations as independent.
    draw_level_probability = (scenario_mixture_winners == strategy_index).mean(axis=0)
    probability = float(draw_level_probability.mean())
    cluster_mcse = float(draw_level_probability.std(ddof=1) / np.sqrt(N_PSA))
    mapping_psa_rows.append(
        ("equal-weight scenario mixture", strategy, probability,
         cluster_mcse, float(scenario_mixture_nb[:, :, strategy_index].mean()))
    )
mapping_psa = pd.DataFrame(
    mapping_psa_rows,
    columns=["Closure mapping", "Strategy", "Probability optimal", "Monte Carlo SE", "Mean illustrative NMB"],
)
print("\nProbability optimal under alternative PPD-to-closure mappings (same random draws):")
print(mapping_psa.pivot(index="Strategy", columns="Closure mapping", values="Probability optimal").round(3).to_string())
print("  Equal mapping weights are a transparent scenario mixture, not posterior model probabilities.")
print("  Mixture Monte Carlo SE treats each PSA draw as a cluster across correlated mappings.")

evpi_boot_rng = np.random.default_rng(RNG_SEED + 991)
evpi_boot = []
for _ in range(200):
    boot_index = evpi_boot_rng.integers(0, N_PSA, N_PSA)
    boot_nb = NB0[boot_index]
    evpi_boot.append(max(float(boot_nb.max(1).mean() - boot_nb.mean(0).max()), 0.0))
EVPI_MCSE = float(np.std(evpi_boot, ddof=1))
print(f"EVPI Monte Carlo SE (200 nonparametric resamples) = ${EVPI_MCSE:,.0f}/site")




In [ ]:
# ---- One-way sensitivity (tornado) on the base-case winner's INMB vs SRP ----
strat_col = {"SRP alone": C["srp"], "SRP+Atridox": C["atridox"],
             "SRP+Periochip": C["periochip"], "SRP+Arestin": C["arestin"]}
BASE = dict(wmd_Atridox=WMD_MEAN["SRP+Atridox"], wmd_Periochip=WMD_MEAN["SRP+Periochip"],
            wmd_Arestin=WMD_MEAN["SRP+Arestin"], hm_base=0.0, cost_mult=1.0, recur=1.0, wtp=WTP0)
_cl = {r.Strategy: (r.PPD_lo, r.PPD_hi) for r in clinical.itertuples() if r.Strategy != "SRP alone"}
RANGES = dict(
    wmd_Periochip=_cl["SRP+Periochip"], wmd_Atridox=_cl["SRP+Atridox"], wmd_Arestin=_cl["SRP+Arestin"],
    hm_base=(0.0, 0.0),
    cost_mult=(1-1.96*0.15, 1+1.96*0.15), recur=(0.76, 1.24), wtp=(20000.0, 100000.0))
PRETTY = {"wmd_Periochip": "CHX PPD effect (mm)", "wmd_Atridox": "DOX PPD effect (mm)",
          "wmd_Arestin": "MINO PPD effect (mm)", "hm_base": "Host-mod QALY", "cost_mult": "Cost multiplier",
          "recur": "Recurrence multiplier", "wtp": "WTP ($/QALY)"}
def nb_at(strat, B):  # HOMOGENEOUS (representative treated-site reference case) - used by the Fig-7E tornado
    w = 0.0 if strat == "SRP alone" else B["wmd_"+KEY_OF[strat]]
    nb, _, _, _ = arm_outcome(strat, w, B["hm_base"], B["cost_mult"], B["recur"], B["wtp"])
    return nb
def nb_pop_at(strat, B, resist=0.0):  # SUBGROUP-PREVALENCE-WEIGHTED - used by all of Figure 8
    add = resist if strat in ("SRP+Atridox", "SRP+Arestin") else 0.0
    tot = 0.0
    for sg in SUBGROUPS:
        w = 0.0 if strat == "SRP alone" else B["wmd_"+KEY_OF[strat]]
        closure = float(np.clip(closure_prob(w) + closure_delta(strat, sg), 0.05, 0.95))
        m = markov(closure, tx_cost=(CFG["COURSE_COST"][strat]+add)*B["cost_mult"],
                   recur_mult=B["recur"], maint=CFG["MAINT_COST"]*B["cost_mult"])
        hm = hm_utility(strat, sg, hm_base=B["hm_base"])
        tot += prev[sg]*(B["wtp"]*(m["qaly"]+hm) - m["cost"])
    return tot
def tornado(target, ref, params):
    base_v = nb_at(target, BASE) - nb_at(ref, BASE); rows = []
    for p in params:
        lo = dict(BASE); lo[p] = RANGES[p][0]; hi = dict(BASE); hi[p] = RANGES[p][1]
        v_lo = nb_at(target, lo) - nb_at(ref, lo); v_hi = nb_at(target, hi) - nb_at(ref, hi)
        rows.append((p, v_lo, v_hi))
    rows.sort(key=lambda r: abs(r[2]-r[1]))
    return base_v, rows
tor_params = ["wmd_Periochip", "hm_base", "cost_mult", "recur", "wtp", "wmd_Atridox"]
tor_base, tor_rows = tornado("SRP+Periochip", "SRP alone", tor_params)
print(f"\nTornado (Periochip INMB vs SRP), base INMB=${tor_base:,.0f}/site; widest driver: "
      f"{PRETTY[tor_rows[-1][0]]}")



In [ ]:
# ---- FIGURE 6: population cost-effectiveness, BIA, CEAC, EVPPI, tornado, PSA cloud ----
fig6 = plt.figure(figsize=(15.5, 9.2))
gs = GridSpec(2, 3, figure=fig6, hspace=0.36, wspace=0.30)

# (A) cost-effectiveness plane + efficient frontier
axA = fig6.add_subplot(gs[0, 0])
for s in STRATS:
    axA.scatter(arms[s]["qaly"], arms[s]["cost"], s=90, color=strat_col[s], zorder=5,
                edgecolor="white", lw=1.2, label=s.replace("SRP alone","SRP").replace("SRP+","+"))
    axA.annotate(s.replace("SRP alone","SRP").replace("SRP+","+"),
                 (arms[s]["qaly"], arms[s]["cost"]), textcoords="offset points",
                 xytext=(6, 5), fontsize=7.5)
fr = frontier.sort_values("QALY")
axA.plot(fr["QALY"], fr["Cost"], "-", color="k", lw=1.3, zorder=3, label="Efficient frontier")
q0 = arms["SRP alone"]["qaly"]; c0 = arms["SRP alone"]["cost"]
qq = np.array([q0-0.02, max(a["qaly"] for a in arms.values())+0.02])
axA.plot(qq, c0 + WTP0*(qq-q0), "--", color=C["mic"], lw=1, label=f"WTP ${WTP0/1000:.0f}k slope")
axA.set_xlabel("Discounted QALYs"); axA.set_ylabel("Discounted cost ($)")
axA.set_title("CE plane - REFERENCE CASE / representative site (frontier: SRP->Periochip @ ~$2,800)", fontsize=9.5)
axA.legend(fontsize=6.8, loc="upper left"); panel_tag(axA, "A")

# (B) two-scenario budget-impact
axB = fig6.add_subplot(gs[0, 1])
adj = [s for s in STRATS if s != "SRP alone"]; xp = np.arange(len(adj)); w = 0.38
net_markov = [bia[s]["net_base"] for s in adj]; net_v1 = [bia[s]["net_v1"] for s in adj]
axB.bar(xp-w/2, net_markov, w, color=C["accent"], label="Markov-derived (this work)")
axB.bar(xp+w/2, net_v1, w, color=C["mic"], label="v1 hard-coded $520 saving")
axB.axhline(0, color="k", lw=0.9)
for i, s in enumerate(adj):
    axB.text(i-w/2, net_markov[i]+8, f"+${net_markov[i]:.0f}", ha="center", fontsize=7, color=C["accent"])
    axB.text(i+w/2, net_v1[i]-18, f"${net_v1[i]:.0f}", ha="center", fontsize=7, color=C["mic"])
axB.set_xticks(xp); axB.set_xticklabels([s.replace("SRP+","") for s in adj])
axB.set_ylabel("Net budget impact ($/treated site)")
axB.set_title("BIA: sign now agrees with CEA (cost-incurring)")
axB.legend(fontsize=7, loc="lower right"); panel_tag(axB, "B")

# (C) cost-effectiveness acceptability curves
axC = fig6.add_subplot(gs[1, 0])
for s in STRATS:
    axC.plot(WTP_GRID/1000, ceac_df[s].values, color=strat_col[s], lw=2,
             label=s.replace("SRP alone","SRP").replace("SRP+","+"))
axC.axvline(WTP0/1000, ls=":", color=C["mic"], lw=1)
axC.set_xlabel("WTP ($000/QALY)"); axC.set_ylabel("P(strategy = optimal)")
axC.set_ylim(-0.02, 1.02)
axC.set_title("CEAC - SUBGROUP-WEIGHTED (each vs ALL comparators, not pairwise vs SRP)", fontsize=9.5)
axC.legend(fontsize=7, loc="center right"); panel_tag(axC, "C")

# (D) EVPPI by parameter group + EVPI reference
axD = fig6.add_subplot(gs[1, 1])
items = sorted(evppi.items(), key=lambda kv: kv[1]); labels = [k for k, _ in items]; vals = [v for _, v in items]
yp = np.arange(len(labels))
axD.barh(yp, vals, color=C["cool"])
axD.axvline(EVPI, ls="--", color=C["accent"], lw=1.2, label=f"EVPI ${EVPI:,.0f}")
for i, v in enumerate(vals):
    axD.text(v+8, i, f"${v:,.0f} ({100*v/EVPI:.0f}%)", va="center", fontsize=7)
axD.set_yticks(yp); axD.set_yticklabels(labels, fontsize=8); axD.set_xlabel("EVPPI ($/treated site)")
axD.set_xlim(0, EVPI*1.28)
axD.set_title("Partial EVPI - SUBGROUP-WEIGHTED (groups not additive)", fontsize=9.5)
axD.legend(fontsize=7.5, loc="lower right"); panel_tag(axD, "D")

# (E) tornado on Periochip INMB vs SRP
axE = fig6.add_subplot(gs[0, 2])
yp = np.arange(len(tor_rows))
for i, (p, v_lo, v_hi) in enumerate(tor_rows):
    left, right = min(v_lo, v_hi), max(v_lo, v_hi)
    axE.barh(i, right-left, left=left, color=C["atridox"], alpha=0.85)
axE.axvline(tor_base, color="k", lw=1.1, ls="--", label=f"base INMB ${tor_base:,.0f}")
axE.set_yticks(yp); axE.set_yticklabels([PRETTY[p] for p, _, _ in tor_rows], fontsize=8)
axE.set_xlabel("Periochip INMB vs SRP ($/treated site)")
axE.set_title("One-way sensitivity - REFERENCE CASE (tornado)", fontsize=9.5)
axE.legend(fontsize=7.5, loc="lower right"); panel_tag(axE, "E")

# (F) PSA cloud on the incremental plane (two contenders vs SRP), subgroup-weighted
axF = fig6.add_subplot(gs[1, 2])
_dCmax = 0.0
for s, col in [("SRP+Periochip", C["periochip"]), ("SRP+Atridox", C["atridox"])]:
    dQ = psa["Q_"+s].values - psa["Q_SRP alone"].values
    dC = psa["C_"+s].values - psa["C_SRP alone"].values
    axF.scatter(dQ, dC, s=6, alpha=0.18, color=col, label=s.replace("SRP+","+"))
    _dCmax = max(_dCmax, np.percentile(dC, 99))
dqx = np.array([-0.02, 0.55]); axF.plot(dqx, WTP0*dqx, "--", color=C["mic"], lw=1, label=f"WTP ${WTP0/1000:.0f}k")
axF.axhline(0, color="k", lw=0.6); axF.axvline(0, color="k", lw=0.6)
axF.set_ylim(-150, max(_dCmax*1.35, 600))          # zoom to the actual cost cloud (was dwarfed by WTP line)
axF.set_xlabel("Incremental QALYs vs SRP"); axF.set_ylabel("Incremental cost vs SRP ($)")
axF.set_title("PSA cloud - SUBGROUP-WEIGHTED (y zoomed; all pts far below WTP line)", fontsize=9.5)
axF.legend(fontsize=7, loc="upper right"); panel_tag(axF, "F")

fig6.suptitle("Figure 6  |  Stage 6: population CEA - REFERENCE CASE (A,B,E) vs SUBGROUP-WEIGHTED "
              "target population (C,D,F); BIA & EVPPI",
              y=1.01, fontsize=11.5, fontweight="bold", ha="left", x=0.02)
save_fig(fig6, "Figure06_population_CEA_PSA_EVPPI")
print("Saved Figure 6")



In [ ]:
# =====================================================================================
# 8 (compute) - DECISION-LEVEL analyses: winner-flip thresholds, decision tornado,
#   probabilistic EVIC, antibiotic-stewardship trade-off, 2-D decision map (P0-3 + P1)
# =====================================================================================
# All Figure-8 analyses use the SUBGROUP-PREVALENCE-WEIGHTED net benefit (nb_pop_at), the same
# lens as Stage 7 and the PSA above - so P(optimal), the winner-flip and the decision map agree.
# (i) WINNER-FLIP: sweep the CHX (Periochip) PPD effect; the subgroup-weighted NB(Atridox) is
#     invariant to it (Atridox closure does not depend on the CHX effect). Published CHX effect
#     is explored between the FDA pivotal ~0.30-mm increment (9 months) and the Ma-Diao
#     0.75-mm meta-analytic estimate (6 months). This is an evidence-scenario range, not a CI.
wmd_sweep = np.linspace(0.30, 1.15, 90)
nb_peri_sweep = np.array([nb_pop_at("SRP+Periochip", {**BASE, "wmd_Periochip": w}) for w in wmd_sweep])
nb_atri_flat = nb_pop_at("SRP+Atridox", BASE)
_diff = nb_peri_sweep - nb_atri_flat
_cross = np.where(np.diff(np.sign(_diff)))[0]
wmd_flip = float(np.interp(0, _diff[[_cross[0], _cross[0]+1]],
                           wmd_sweep[[_cross[0], _cross[0]+1]])) if len(_cross) else np.nan
CHX_EVIDENCE_RANGE = (0.30, 0.75); CHX_CI_HI = 0.75
print("Winner-flip analysis (subgroup-weighted, Periochip vs Atridox, WTP ${:,.0f}):".format(WTP0))
if wmd_flip == wmd_flip and wmd_flip <= CHX_CI_HI:
    print(f"  winner flips at CHX PPD effect ~= {wmd_flip:.2f} mm (within the cross-design evidence-scenario range 0.30-0.75 mm): "
          f"Atridox wins below it, Periochip above.")
else:
    print(f"  Atridox wins across the evidence-scenario range (0.30-0.75 mm); Periochip would need "
          f"CHX ~= {wmd_flip:.2f} mm to overtake - beyond the evidence. In the subgroup-weighted target "
          f"population the choice (Atridox) is robust to the CHX effect size.")

# (ii) DECISION TORNADO: INMB(Periochip - Atridox); which arm wins at each parameter's lo/hi.
def inmb_pa(B): return nb_pop_at("SRP+Periochip", B) - nb_pop_at("SRP+Atridox", B)
dt_params = ["wmd_Periochip", "hm_base", "wmd_Atridox", "recur", "cost_mult", "wtp"]
dt_base = inmb_pa(BASE); dt_rows = []
for p in dt_params:
    lo = dict(BASE); lo[p] = RANGES[p][0]; hi = dict(BASE); hi[p] = RANGES[p][1]
    dt_rows.append((p, inmb_pa(lo), inmb_pa(hi)))
dt_rows.sort(key=lambda r: abs(r[2]-r[1]))
flips = [PRETTY[p] for p, a, b in dt_rows if (min(a, b) < 0 < max(a, b))]
print(f"  Decision tornado: base INMB(Peri-Atri)=${dt_base:,.0f} ({'Periochip' if dt_base>=0 else 'Atridox'} wins at base); "
      f"parameters that can still FLIP the winner within their CI: {flips if flips else 'none'}")

# (iii) PROBABILISTIC EVIC (computed in Stage 6c on the full N={} shared sample). Reported with an
#       FP-robust indicator - the fraction of draws in which subgroup stratification actually
#       changes the chosen strategy (identical to P(EVIC>0) but immune to boundary rounding).
p_strat = float(np.mean(strat_matters))
print(f"  Probabilistic EVIC: mean ${evic_draws.mean():,.0f}/site "
      f"(95% CI ${np.percentile(evic_draws,2.5):,.0f}-${np.percentile(evic_draws,97.5):,.0f}); "
      f"deterministic EVIC ${EVIC:,.0f}. P(stratification changes the choice)={p_strat:.2f}.".format(N_PSA))

# (iv) ANTIBIOTIC-STEWARDSHIP trade-off: an expected per-course resistance COST on the antibiotic
#     arms only (CHX is a biocide, no resistance selection). Caton 2000 found NO doxycycline
#     resistance with sub-antimicrobial dosing -> plausible penalty ~ $0; sweep to locate crossover.
res_sweep = np.linspace(0, 1000, 90)
stew = {}
for chx, tag in [(0.75, "CHX 0.75 (meta-analysis)"), (0.30, "CHX 0.30 (FDA pivotal)")]:
    B = {**BASE, "wmd_Periochip": chx}
    nb_p = nb_pop_at("SRP+Periochip", B, 0.0)
    stew[tag] = np.array([nb_pop_at("SRP+Atridox", B, r) - nb_p for r in res_sweep])
r_cross = {tag: (float(np.interp(0, arr[::-1], res_sweep[::-1])) if (arr.min() < 0 < arr.max()) else np.nan)
           for tag, arr in stew.items()}
_rc = r_cross["CHX 0.75 (meta-analysis)"]; _rc_s = f"~${_rc:.0f}/course" if _rc == _rc else ">$1000/course"
# The resistance penalty falls only on the antibiotic arms. At the base CHX effect (0.75 mm)
# Periochip already leads, so raising the penalty only widens its lead; a penalty changes the
# winner only in the below-flip regime (e.g. FDA pivotal ~0.30 mm), where it erodes the gel's lead.
print(f"  Stewardship (subgroup-weighted): at the base CHX effect (0.75 mm) Periochip already leads, "
      f"so the antibiotic resistance penalty only widens its lead; a penalty changes the winner only in "
      f"the below-flip regime (e.g. FDA pivotal ~0.30 mm), where it erodes the doxycycline gel's lead. "
      f"Caton 2000 implies ~$0 doxycycline resistance under sub-antimicrobial dosing.")

# (v) 2-D DECISION MAP over CHX effect x host-modulation QALY -> argmax strategy
wg = np.linspace(0.35, 0.95, 55); hg = np.linspace(0.0, 0.10, 55)
Zmap = np.zeros((len(hg), len(wg)), int)
for iy, h in enumerate(hg):
    for ix, w in enumerate(wg):
        B = {**BASE, "wmd_Periochip": w, "hm_base": h}
        Zmap[iy, ix] = int(np.argmax([nb_pop_at(s, B) for s in STRATS]))
present_strats = sorted(set(Zmap.flatten()))
print("  2-D decision map computed (strategies appearing as optimal:",
      [STRATS[i] for i in present_strats], ")")



In [ ]:
# ---- FIGURE 8: DECISION-LEVEL analyses (winner-flip, decision tornado, P(optimal),
#      probabilistic EVIC, stewardship trade-off, 2-D decision map) ----
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
fig8 = plt.figure(figsize=(15.5, 9.2))
gs = GridSpec(2, 3, figure=fig8, hspace=0.38, wspace=0.30)

# (A) winner-flip curve
axA = fig8.add_subplot(gs[0, 0])
axA.plot(wmd_sweep, nb_peri_sweep/1000, color=C["periochip"], lw=2.2, label="Periochip NB")
axA.axhline(nb_atri_flat/1000, color=C["atridox"], lw=2.0, ls="-", label="Atridox NB (CHX-invariant)")
axA.axvspan(CHX_EVIDENCE_RANGE[0], CHX_EVIDENCE_RANGE[1], color=C["accent"], alpha=0.10, label="evidence scenario 0.30-0.75")
axA.axvline(0.30, ls=":", color=C["accent"], lw=1.2, label="FDA pivotal ~0.30")
axA.axvline(0.75, ls=":", color=C["mic"], lw=1.2); axA.text(0.755, axA.get_ylim()[0], "Ma&Diao\n0.75", fontsize=6.6, color=C["mic"])
if wmd_flip == wmd_flip:
    axA.axvline(wmd_flip, ls="--", color="k", lw=1.1)
    axA.text(wmd_flip+0.01, nb_atri_flat/1000, f"flip {wmd_flip:.2f}", ha="left", fontsize=7.5, fontweight="bold")
axA.set_xlabel("CHX (Periochip) PPD effect (mm)"); axA.set_ylabel("Subgroup-wtd population NB ($000/site)")
axA.set_title(f"Winner-flip at CHX {wmd_flip:.2f} mm (base 0.75 mm: chip wins)", fontsize=10)
axA.legend(fontsize=6.8, loc="lower right"); panel_tag(axA, "A")

# (B) decision tornado (Periochip - Atridox)
axB = fig8.add_subplot(gs[0, 1])
for i, (p, v_lo, v_hi) in enumerate(dt_rows):
    left, right = min(v_lo, v_hi), max(v_lo, v_hi)
    if left < 0 < right:   # spans the flip -> split-colour
        axB.barh(i, 0-left, left=left, color=C["atridox"], alpha=0.85)
        axB.barh(i, right-0, left=0, color=C["periochip"], alpha=0.85)
    else:
        axB.barh(i, right-left, left=left, color=(C["periochip"] if left >= 0 else C["atridox"]), alpha=0.85)
axB.axvline(0, color="k", lw=1.2); axB.axvline(dt_base, color=C["mic"], ls="--", lw=1, label=f"base ${dt_base:,.0f}")
axB.set_yticks(range(len(dt_rows))); axB.set_yticklabels([PRETTY[p] for p, _, _ in dt_rows], fontsize=8)
axB.set_xlabel("INMB Periochip - Atridox ($/site)   [<0: Atridox wins]")
axB.set_title("Decision tornado - subgroup-weighted (which arm wins)", fontsize=10)
axB.legend(fontsize=7.5, loc="lower right"); panel_tag(axB, "B")

# (C) P(each strategy optimal)
axC = fig8.add_subplot(gs[0, 2])
order = ["SRP+Periochip", "SRP+Atridox", "SRP+Arestin", "SRP alone"]
vals = [p_optimal[s] for s in order]
axC.bar(range(len(order)), vals, color=[strat_col[s] for s in order])
for i, v in enumerate(vals): axC.text(i, v+0.012, f"{v:.2f}", ha="center", fontsize=8, fontweight="bold")
axC.set_xticks(range(len(order))); axC.set_xticklabels([s.replace("SRP alone","SRP").replace("SRP+","+") for s in order], fontsize=8)
axC.set_ylabel(f"P(optimal) @ WTP ${WTP0/1000:.0f}k"); axC.set_ylim(0, 1.0)
axC.set_title(f"P(optimal), subgroup-weighted: Periochip {p_optimal['SRP+Periochip']:.2f} > Atridox {p_optimal['SRP+Atridox']:.2f}", fontsize=10); panel_tag(axC, "C")

# (D) probabilistic EVIC distribution
axD = fig8.add_subplot(gs[1, 0])
axD.hist(evic_draws, bins=40, color=C["cool"], alpha=0.85, edgecolor="white")
axD.axvline(evic_draws.mean(), color=C["accent"], lw=1.6, label=f"prob. mean ${evic_draws.mean():,.0f}")
axD.axvline(EVIC, color="k", ls="--", lw=1.4, label=f"deterministic ${EVIC:,.0f}")
axD.set_xlabel("EVIC ($/treated site)"); axD.set_ylabel("PSA draws")
axD.set_title(f"Value of stratification (P(strat. changes choice)={p_strat:.2f})", fontsize=10)
axD.legend(fontsize=7.3, loc="upper right"); panel_tag(axD, "D")

# (E) antibiotic-stewardship trade-off
axE = fig8.add_subplot(gs[1, 1])
for tag, arr in stew.items():
    axE.plot(res_sweep, arr/1000, lw=2, label=tag)
axE.axhline(0, color="k", lw=1.0)
if _rc == _rc:
    axE.axvline(_rc, ls="--", color="k", lw=1.0)
    axE.text(_rc+12, 0.05, f"Periochip overtakes\nonly at ~${_rc:.0f}", fontsize=6.8)
axE.axvspan(0, 30, color=C["ok"], alpha=0.12); axE.text(34, axE.get_ylim()[1]*0.78, "Caton: ~$0\n(no resistance)", fontsize=6.8, color=C["ok"])
axE.set_xlabel("Per-course antibiotic resistance cost ($)")
axE.set_ylabel("NB Atridox - Periochip ($000/site)  [>0: Atridox wins]")
axE.set_title("Stewardship: Periochip leads at base; penalty matters only below the flip", fontsize=10)
axE.legend(fontsize=7.3, loc="upper right"); panel_tag(axE, "E")

# (F) 2-D decision map
axF = fig8.add_subplot(gs[1, 2])
cmap4 = ListedColormap([strat_col[s] for s in STRATS])
axF.imshow(Zmap, origin="lower", aspect="auto", cmap=cmap4, vmin=0, vmax=len(STRATS)-1,
           extent=[wg[0], wg[-1], hg[0], hg[-1]])
axF.axvspan(CHX_EVIDENCE_RANGE[0], CHX_EVIDENCE_RANGE[1], color="white", alpha=0.0)
axF.plot([CHX_EVIDENCE_RANGE[0], CHX_EVIDENCE_RANGE[0]], [hg[0], hg[-1]], "w:", lw=1); axF.plot([CHX_EVIDENCE_RANGE[1], CHX_EVIDENCE_RANGE[1]], [hg[0], hg[-1]], "w:", lw=1)
axF.axvline(0.75, color="white", ls=":", lw=1)
axF.scatter([0.75], [0.05], color="white", edgecolor="k", s=45, zorder=5)
axF.set_xlabel("CHX (Periochip) PPD effect (mm)"); axF.set_ylabel("Host-modulation QALY")
axF.set_title("Decision map - subgroup-weighted (colour = optimal)", fontsize=10)
axF.legend(handles=[Patch(color=strat_col[STRATS[i]], label=STRATS[i].replace("SRP+","+"))
                    for i in present_strats], fontsize=7.3, loc="lower right")
panel_tag(axF, "F")

fig8.suptitle("Figure 8  |  Stage 7-8: decision-level robustness (SUBGROUP-WEIGHTED target population) "
              "- winner-flip, stewardship, value of information",
              y=1.01, fontsize=11.5, fontweight="bold", ha="left", x=0.02)
save_fig(fig8, "Figure08_decision_robustness")
print("Saved Figure 8")



In [ ]:
# =====================================================================================
# REPRODUCIBILITY MANIFEST (reviewer method#1) - seeds, sample sizes, distributions, versions
# =====================================================================================
import sys, platform, sklearn, scipy
print("REPRODUCIBILITY MANIFEST")
print("-"*80)
print(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} "
      f"| scipy {scipy.__version__} | sklearn {sklearn.__version__} | matplotlib {mpl.__version__}")
print(f"Global RNG seed: {RNG_SEED} (numpy default_rng)")
print(f"Sobol GSA: n_base=1024 + 300 bootstrap (~{(2+len(gsa_names))*1024} PBPK ODE solves), LSODA rtol=1e-6")
print(f"PSA: N={N_PSA} draws | EVPPI: Strong-Oakley-Brennan 2014 GAM "
      f"(cubic SplineTransformer, 6 knots) + LinearRegression, five-fold cross-fitted out-of-sample")
print(f"Markov: {CFG['HORIZON_Y']}-y horizon, {CFG['DISCOUNT']:.0%} discount, 4 states, "
      f"utilities {list(CFG['UTIL'])}, unit = 1 treated periodontal site (N_INDEX_SITES={CFG['N_INDEX_SITES']})")
print("PSA parameter distributions:")
print("  PPD WMD Atridox/Arestin: truncated Normal(mean, SD from published 95% CI)")
print("  PPD WMD PerioChip      : Uniform(0.30, 0.75) evidence scenario; not a statistical CI")
print(f"  Cross-product WMD correlation: rho={RHO_WMD} (no empirical covariance available)")
print("  host-mod base QALY    : 0 in evidence-audited base (no empirical local-LDD -> QALY map)")
print("  cost multiplier       : Gamma(mean 1, CV 15%)")
print("  recurrence multiplier : Normal(1, 0.12) clipped [0.7, 1.6]")
if CFG["PROPAGATE_STRUCTURAL_PSA"]:
    for k, (m, s, lo, hi) in CFG["STRUCT_PSA"].items():
        print(f"  {k:12s}        : truncated Normal(mean {m}, SD {s}) on [{lo}, {hi}]  (reviewer #6 structural)")
print(f"Host-modulation config : pathway='{CFG['HM_PATHWAY']}', scenario='{CFG['HM_SCENARIO']}' "
      f"(strength x{CFG['HM_SCEN_MULT'][CFG['HM_SCENARIO']]}); single-pathway rule enforced (reviewer #2).")
print(f"Closure mapping        : closure = clip({CFG['P0_SRP']} + {CFG['KAPPA_CLOSE']}*PPD_WMD, 0.05, "
      f"{CFG['PCLOSE_CEIL']}); Hill gradient gamma={GAMMA_CL} (reviewer #6).")
print("-"*80)

print(f"Structural mapping scenarios: {', '.join(CFG['CLOSURE_MAPPINGS'])}; base={CFG['CLOSURE_MAPPING']}")
print("Mapping robustness: same 5,000 PSA draws reused for every functional form; equal-weight mixture is not Bayesian.")
print(f"EVPI Monte Carlo SE: ${EVPI_MCSE:,.0f}/site from 200 nonparametric resamples")
print("EVPPI: five-fold out-of-sample metamodel predictions; reported values constrained to [0, EVPI]")
print("PK calibration identifiability: multistart solutions + residual-Jacobian rank/condition diagnostics")
print("Independent PK/GCF validation: not performed (no independent patient-level dataset available)")




In [ ]:
# =====================================================================================
# APPENDIX - export tables in the MANUSCRIPT index order
#   Main tables:          Table 1 evidence base | Table 2 base-case CEA | Table 3 robustness
#   Supplementary tables: S1 PBPK params | S2 PK/PD indices | S3 calibration status |
#                         S4 budget impact | S5 value of information
# =====================================================================================
import os
os.makedirs(TABDIR, exist_ok=True)
from docx import Document
from docx.shared import Pt
from docx.enum.section import WD_ORIENT

# ---------- assemble table content from live objects ----------
# Table 1 (main): standardized evidence base
t1 = clinical[["Strategy","PPD_WMD","CAL_WMD","k_studies","ClosureProb","Evidence_note","Source"]].copy()
t1.columns = ["Strategy","PPD WMD (mm)","CAL WMD (mm)","k RCTs","Closure proxy (structural)","Pivotal finding","Source"]
for column in ["PPD WMD (mm)", "CAL WMD (mm)", "Closure proxy (structural)"]:
    t1[column] = t1[column].round(3)

# Table 2 (main): base-case lifetime cost-effectiveness
t2 = ce.copy(); t2["ICER_vs_SRP"] = t2["ICER_vs_SRP"].map(lambda x: f"{x:,.0f}" if x==x else "-")
t2["Cost"] = t2["Cost"].round(0)
t2["QALY"] = t2["QALY"].round(3)
t2["ToothYrs"] = t2["ToothYrs"].round(3)
base_display = t2.loc[t2["Arm"].eq("SRP alone")].iloc[0]
t2["dCost"] = t2["Cost"] - base_display["Cost"]
t2["dQALY"] = (t2["QALY"] - base_display["QALY"]).round(3)
t2["dToothYrs"] = (t2["ToothYrs"] - base_display["ToothYrs"]).round(3)
t2["ICER_vs_SRP"] = np.where(t2["dQALY"].abs() > 1e-12, t2["dCost"] / t2["dQALY"], np.nan)
t2["ICER_vs_SRP"] = t2["ICER_vs_SRP"].map(lambda x: f"{x:,.0f}" if x == x else "-")
icer_interval_by_arm = {
    row["Arm"]: (
        f"{row['ICER 2.5th (positive dQALY draws)']:,.0f}–"
        f"{row['ICER 97.5th (positive dQALY draws)']:,.0f}"
    )
    for _, row in reference_case_psa_icer_ranges.iterrows()
}
t2["PSA_ICER_95_interval"] = t2["Arm"].map(icer_interval_by_arm).fillna("-")
t2.columns = ["Arm","Illustrative cost ($/site)","QALY/site","Tooth-years/site","Illustrative dCost ($/site)","dQALY/site","dTooth-years/site","Display-consistent ICER vs SRP ($/QALY)","PSA ICER 2.5th–97.5th percentiles ($/QALY)*"]

# Table 3 (main): decision-level robustness (chip-wins narrative, matches computed p_optimal)
t3_rows = [
    ("Reference-case winner (homogeneous, representative treated site)",
     "SRP+Periochip is the illustrative frontier strategy under the conservative initial-state-only coupling (display-consistent ICER approximately $2,800/QALY)"),
    ("Subgroup-weighted population-optimal",
     f"{pop_best_strat} (optimal in every subgroup; no host-modulation flip at the base "
     f"chlorhexidine effect; value of host-modulation ${value_of_HM:,.0f}/site)"),
    ("P(optimal) @ WTP $50k/QALY  [subgroup-weighted PSA]",
     f"Periochip {p_optimal['SRP+Periochip']:.2f}, Atridox {p_optimal['SRP+Atridox']:.2f}, "
     f"Arestin {p_optimal['SRP+Arestin']:.2f}, SRP {p_optimal['SRP alone']:.2f}"),
    ("Chlorhexidine-effect winner-flip threshold (subgroup-weighted)",
     f"{wmd_flip:.2f} mm: PerioChip wins above it in this structural model; the explored CHX evidence scenario is "
     "0.30 mm (FDA pivotal, 9 months) to 0.75 mm (Ma-Diao meta-analysis, 6 months)"),
    ("Parameters that can flip the winner within CI",
     ", ".join(flips) if flips else "none"),
    ("EVIC deterministic vs probabilistic",
     f"${EVIC:,.0f} vs mean ${evic_draws.mean():,.0f} (95% CI ${np.percentile(evic_draws,2.5):,.0f}-"
     f"${np.percentile(evic_draws,97.5):,.0f}); P(stratification changes the choice)={p_strat:.2f}"),
    ("Antibiotic-resistance penalty (chip already leads at CHX 0.75 mm)",
     "not required at the base chlorhexidine effect - the chip already leads; a penalty is relevant "
     "only in a low-effect scenario, where it can erode the doxycycline gel's lead"),
]
t3 = pd.DataFrame(t3_rows, columns=["Decision analysis","Result"])

# Table S1 (supp): mini-PBPK parameters
pbpk_rows = [
    ("Depot","R0",f"{PBPK['R0']:.3f}","ug/h","zero-order release term","derived from the fixed 1-mg/14-day ARESTIN label summary"),
    ("Pocket fluid","Vpf",f"{PBPK['Vpf']}","uL","effective pocket-fluid volume","structural assumption; not externally identified"),
    ("Pocket fluid","CL_eff",f"{PBPK['CL_eff']}","uL/h","effective local loss/turnover","calibrated nuisance parameter; non-identifiable"),
    ("Biofilm","Vbf",f"{PBPK['Vbf']}","uL","effective biofilm volume","calibrated nuisance parameter; non-identifiable"),
    ("Biofilm","PSbf",f"{PBPK['PSbf']}","uL/h","pocket-biofilm exchange","calibrated nuisance parameter; non-identifiable"),
    ("Tissue","Vti",f"{PBPK['Vti']}","uL","effective gingival-tissue volume","calibrated nuisance parameter; non-identifiable"),
    ("Tissue","PSti",f"{PBPK['PSti']}","uL/h","pocket-tissue exchange","calibrated nuisance parameter; non-identifiable"),
    ("Tissue","Kp",f"{PBPK['Kp']}","-","tissue:pocket partition","calibrated nuisance parameter; non-identifiable"),
    ("Absorbed sink","kabs",f"{PBPK['kabs']}","1/h","tissue-to-absorbed-sink rate","calibrated nuisance parameter; not human systemic PK"),
]
s1 = pd.DataFrame(pbpk_rows, columns=["Compartment","Parameter","Value","Unit","Meaning","Source"])

# Table S2 (supp): PK/PD indices
s2 = pkpd_tbl[["Product","Cmax","Threshold","Cmax_thr","AUC_thr","Tabove_d","pctTabove"]].copy()
s2.columns = ["Product","Cmax (µg/mL)","Threshold (µg/mL)","Cmax/thr","AUC/thr","T>thr (d)","%T>thr"]

# Table S3 (supp): calibration to published summary anchors
core_lbl = {"zero":"zero-order (microsphere)","first":"first-order","biphasic":"biphasic burst+sustained"}
s3_rows = []
for k in PK_ANCHOR_KEYS:
    f = prod_fits[k]
    hold = "not performed"
    s3_rows.append((PK_ANCHORS[k]["product"], core_lbl[f["rtype"]], f["n_tot"],
                    f"{f['aafe_full']:.2f}", hold, PK_ANCHORS[k]["src"]))
s3 = pd.DataFrame(s3_rows, columns=["Product","Release core","GCF points","Full-profile AAFE",
                                    "Independent validation","Source"])

# Table S4 (supp): two-scenario budget impact
s4_rows = []
for s in STRATS:
    if s == "SRP alone": continue
    b = bia[s]
    s4_rows.append((s, f"{b['course']:.0f}", f"{b['save_markov']:.0f}", f"{b['net_base']:+.0f}",
                    f"{POP_TREATED*b['net_base']/1e6:+.2f}M", f"{b['save_breakeven']:.0f}"))
s4 = pd.DataFrame(s4_rows, columns=["Arm","Course cost ($)","Markov downstream saving ($)",
                                    "Net ($/site)","Budget / 10k sites","Break-even saving ($)"])

# Table S5 (supp): value of information
s5_rows = [("EVPI (overall)", f"{EVPI:,.0f}", f"{EVPI*POP_TREATED/1e6:.2f}M", "100")]
for k in ["Closure/efficacy (PPD WMD)", "Closure mapping (P0/kappa/ceiling)",
          "Host-modulation QALY (disabled)", "Recurrence/retention", "Costs"]:
    v = evppi[k]; s5_rows.append((f"EVPPI: {k}", f"{v:,.0f}", f"{v*POP_TREATED/1e6:.2f}M", f"{100*v/EVPI:.0f}"))
s5 = pd.DataFrame(s5_rows, columns=["Metric","Per treated site ($)","Per 10,000 treated sites ($)","% of EVPI"])



# Table S5: release-parameter identifiability diagnostics
s3b = identifiability_tbl.copy()

# Table S6: deterministic alternative-mapping sensitivity
s6 = mapping_sensitivity.copy()
for column in ["SRP closure", "Atridox closure", "PerioChip closure", "Arestin closure"]:
    s6[column] = s6[column].round(3)
s6["INMB PerioChip vs Atridox"] = (s6["INMB PerioChip vs Atridox"] / 100).round() * 100
s6["PerioChip ICER vs SRP"] = (s6["PerioChip ICER vs SRP"] / 100).round() * 100

# Table S7: structural sensitivity to where the closure proxy enters the Markov model
s6b = closure_coupling_sensitivity.copy()
for column in ["SRP closure", "Atridox closure", "PerioChip closure", "Arestin closure"]:
    s6b[column] = s6b[column].round(3)
s6b["INMB PerioChip vs Atridox"] = (s6b["INMB PerioChip vs Atridox"] / 100).round() * 100
s6b["PerioChip ICER vs SRP"] = (s6b["PerioChip ICER vs SRP"] / 100).round() * 100

# Table S8: same-draw alternative-mapping PSA
s7 = mapping_psa.copy()
s7["Probability optimal"] = s7["Probability optimal"].round(3)
s7["Monte Carlo SE"] = s7["Monte Carlo SE"].round(3)
s7["Mean illustrative NMB"] = (s7["Mean illustrative NMB"] / 100).round() * 100

# Table S9: MMP-8 exclusion and mutually exclusive single-pathway scenarios
s8 = host_modulation_sensitivity.copy()
s8["Illustrative NMB"] = (s8["Illustrative NMB"] / 100).round() * 100

# Table S10: complete provenance of the proof-of-concept economic and Markov inputs
utility_display = "[" + ", ".join(f"{float(x):.2f}" for x in CFG["UTIL"]) + "]"
course_cost_display = "; ".join(f"{arm}={float(cost):.0f}" for arm, cost in CFG["COURSE_COST"].items())
subgroup_display = "; ".join(f"{group}={float(value):.2f}" for group, value in CFG["SUBGROUP_PREV"].items())
subgroup_modifier_display = "; ".join(
    f"{group}={float(value):+.2f}" for group, value in CFG["SUBGROUP_CLOSURE_DELTA"].items()
)
def _matrix_display(matrix):
    return "[" + "; ".join(",".join(f"{float(value):.3f}" for value in row) for row in matrix) + "]"

economic_input_provenance = pd.DataFrame([
    ("Analysis unit", "one representative treated periodontal site", "structural assumption", "not patient-level"),
    ("Time horizon", CFG["HORIZON_Y"], "illustrative assumption", "no health-system calibration"),
    ("Discount rate", CFG["DISCOUNT"], "illustrative assumption", "not country-specific"),
    ("WTP threshold", CFG["WTP"], "illustrative assumption", "not jurisdiction-specific"),
    ("Health-state utilities", utility_display, "illustrative assumption", "no empirical oral-utility source"),
    ("Annual maintenance cost", CFG["MAINT_COST"], "illustrative assumption", "no payer source"),
    ("Course costs", course_cost_display, "illustrative assumption", "no payer source"),
    ("Initial state vector: good response", _matrix_display([MARKOV_INITIAL_GOOD]), "illustrative assumption", "not externally calibrated"),
    ("Initial state vector: poor response", _matrix_display([MARKOV_INITIAL_POOR]), "illustrative assumption", "not externally calibrated"),
    ("Markov transition matrix: good response", _matrix_display(MARKOV_T_GOOD), "illustrative assumption", "not externally calibrated"),
    ("Markov transition matrix: poor response", _matrix_display(MARKOV_T_POOR), "illustrative assumption", "not externally calibrated"),
    ("Base closure-Markov coupling", CFG["CLOSURE_MARKOV_COUPLING"], "conservative structural assumption", "closure proxy enters once; transition-only and dual-entry alternatives are in Table S7"),
    ("Health-state cost multipliers", "[0.3, 0.6, 1.0, 0.0] x annual maintenance cost", "illustrative assumption", "not externally calibrated"),
    ("Subgroup prevalences", subgroup_display, "illustrative stress test", "not epidemiologically calibrated"),
    ("Subgroup closure modifiers", subgroup_modifier_display, "illustrative stress test", "not empirically calibrated"),
    ("Atridox/Arestin PPD-effect PSA", "Normal(mean=meta-analytic WMD, SD=CI-derived SE), truncated at 0", "published aggregate evidence", "distributional form is an analytical assumption"),
    ("PerioChip PPD-effect PSA", "Uniform(0.30, 0.75) mm", "cross-design evidence scenario", "bounds are not a sampling CI and are not exchangeable with the other products"),
    ("Treatment-cost multiplier PSA", "Gamma(mean=1, CV=0.15)", "illustrative assumption", "no empirical payer distribution"),
    ("Recurrence multiplier PSA", "Normal(mean=1, SD=0.12), clipped to [0.70, 1.60]", "illustrative assumption", "no empirical longitudinal distribution"),
    ("Closure baseline PSA", "Truncated Normal(mean=0.33, SD=0.05), bounds [0.15, 0.50]", "structural mapping assumption", "no independent pocket-closure calibration dataset"),
    ("Closure slope PSA", "Truncated Normal(mean=0.25, SD=0.06), bounds [0.10, 0.45]", "structural mapping assumption", "no independent pocket-closure calibration dataset"),
    ("Closure ceiling PSA", "Truncated Normal(mean=0.75, SD=0.06), bounds [0.60, 0.90]", "structural mapping assumption", "no independent pocket-closure calibration dataset"),
    ("MMP-8 QALY increment", 0.0, "conservative base case", "no independent MMP-8-to-QALY mapping"),
], columns=["Parameter group", "Value", "Evidence class", "Source/limitation"])
s9 = economic_input_provenance.copy()

# ---------- write CSVs in manuscript numbering ----------
csv_map = [("Table01_evidence_base", t1), ("Table02_base_case_CEA", t2),
           ("Table03_decision_robustness", t3), ("TableS03_PBPK_parameters", s1),
           ("TableS01_PKPD_indices", s2), ("TableS04_PBPK_calibration_status", s3),
           ("TableS05_PBPK_identifiability", s3b), ("TableS11_budget_impact", s4),
           ("TableS12_value_of_information", s5), ("TableS06_closure_mapping_sensitivity", s6),
           ("TableS07_closure_Markov_coupling_sensitivity", s6b),
           ("TableS08_mapping_PSA", s7), ("TableS09_host_modulation_sensitivity", s8),
           ("TableS10_economic_input_provenance", s9),
           ("reference_case_PSA_ICER_ranges", reference_case_psa_icer_ranges)]
PARAMETER_PROVENANCE.to_csv(os.path.join(TABDIR, "TableS02_parameter_provenance.csv"), index=False)
for name, df in csv_map:
    df.to_csv(os.path.join(TABDIR, f"{name}.csv"), index=False)


analysis_summary = {
    "analysis_class": CFG["ANALYSIS_CLASS"],
    "analysis_unit": "one representative treated periodontal site",
    "base_closure_mapping": CFG["CLOSURE_MAPPING"],
    "closure_mapping_scenarios": list(CFG["CLOSURE_MAPPINGS"]),
    "closure_markov_coupling_base": CFG["CLOSURE_MARKOV_COUPLING"],
    "closure_markov_coupling_scenarios": list(CFG["CLOSURE_MARKOV_COUPLING_SCENARIOS"]),
    "host_modulation_base_pathway": CFG["HM_PATHWAY"],
    "host_modulation_base_qaly": CFG["HM_BASE_QALY"],
    "independent_pk_validation": "not performed",
    "psa_draws": N_PSA,
    "reference_case_psa_icer_ranges": reference_case_psa_icer_ranges.to_dict(orient="records"),
    "reference_case_icer_conditioning": "ICER percentiles calculated among draws with positive incremental QALYs; P(dQALY <= 0) reported separately",
    "evpi": EVPI,
    "evpi_mcse": EVPI_MCSE,
    "evppi_estimation_method": "five-fold cross-fitted GAM with out-of-sample predictions",
    "evppi_reporting_bound": "reported values constrained to [0, EVPI]",
    "evppi_raw_cross_fitted": {name: float(value) for name, value in evppi_raw.items()},
    "evppi_reported": {name: float(value) for name, value in evppi.items()},
    "base_probability_optimal": {strategy: float(value) for strategy, value in p_optimal.items()},
    "mapping_probability_optimal": mapping_psa.to_dict(orient="records"),
    "mapping_mixture_mcse_method": "PSA-draw-cluster SE across correlated mappings",
    "closure_coupling_sensitivity": closure_coupling_sensitivity.to_dict(orient="records"),
    "identifiability": identifiability_tbl.to_dict(orient="records"),
}
with open(os.path.join(RESULT, "analysis_summary.json"), "w", encoding="utf-8") as handle:
    json.dump(analysis_summary, handle, ensure_ascii=False, indent=2, default=str)

# ---------- build the Word document (Main tables then Supplementary) ----------
def add_df_table(doc, df, style="Light Grid Accent 1"):
    tbl = doc.add_table(rows=1, cols=len(df.columns)); tbl.style = style
    for j, col in enumerate(df.columns):
        cell = tbl.rows[0].cells[j]; cell.text = str(col)
        runs = cell.paragraphs[0].runs
        if not runs: runs = [cell.paragraphs[0].add_run(str(col))]
        for r in runs: r.font.bold = True; r.font.size = Pt(8.5)
    for _, row in df.iterrows():
        cells = tbl.add_row().cells
        for j, col in enumerate(df.columns):
            cells[j].text = str(row[col])
            for p in cells[j].paragraphs:
                for r in p.runs: r.font.size = Pt(8.5)
    return tbl
def cap(doc, text):
    p = doc.add_paragraph(); r = p.add_run(text); r.font.bold = True; r.font.size = Pt(9.5); return p
def note(doc, text):
    p = doc.add_paragraph(); r = p.add_run(text); r.font.italic = True; r.font.size = Pt(8); return p

doc = Document()
table_section = doc.sections[0]
table_section.orientation = WD_ORIENT.LANDSCAPE
table_section.page_width, table_section.page_height = table_section.page_height, table_section.page_width
doc.add_heading("Main Tables", level=1)

cap(doc, "Table 1. Standardized clinical evidence base with published PPD effects and structural closure proxies.")
add_df_table(doc, t1)
note(doc, "WMD, weighted mean difference vs SRP. Closure probability = 0.33 + 0.25*PPD WMD, capped at 0.75. "
          "The closure column is a structural proxy, not a reported clinical probability. The CHX evidence scenario uses "
          "the FDA pivotal ~0.30-mm increment at 9 months and Ma-Diao 0.75 mm at 6 months; it is not a CI.")

cap(doc, "Table 2. Base-case lifetime cost-effectiveness (payer perspective, 20 years, 3% discount).")
add_df_table(doc, t2)
note(doc, "Efficient-frontier winner: SRP+Periochip (chlorhexidine chip), sequential ICER approximately $2,800/QALY. "
          "Off-frontier adjuncts are strongly (simple) dominated where the chip is both cheaper and more "
          "effective. This antimicrobial-only ranking is confirmed, not overturned, once host-modulation value "
          "and chlorhexidine-effect uncertainty are added (see Table 3 and Table S5). *PSA ICER intervals are "
          "2.5th–97.5th percentiles among draws with positive incremental QALYs; the separate CSV reports the "
          "probability of non-positive incremental QALYs.")

cap(doc, "Table 3. Decision-level robustness and the value of stratification - two clearly separated lenses.")
add_df_table(doc, t3)
note(doc, f"Two lenses are kept distinct: (1) the reference-case (homogeneous, representative treated-site) CEA, frontier "
          f"winner Periochip (ICER approximately $2,800/QALY); and (2) the subgroup-weighted target-population decision, whose "
          f"optimal single strategy is also Periochip (P(optimal) {p_optimal['SRP+Periochip']:.2f} vs "
          f"{p_optimal['SRP+Atridox']:.2f} for the doxycycline gel). The winner flips to the gel only if the "
          f"chlorhexidine effect falls below ~{wmd_flip:.2f} mm (i.e. the low FDA-pivotal evidence scenario), below the "
          f"Ma & Diao 0.75 mm base. Stratification changes the choice in only {p_strat:.0%} of draws, so its "
          "expected value is ~$0.")

doc.add_page_break()
doc.add_heading("Supplementary Tables", level=1)

cap(doc, "Table S1. Local mini-PBPK structural parameters with an absorbed sink (effective volumes uL, amounts ug, time h).")
add_df_table(doc, s1)
note(doc, f"Mass-balance error {mb_err:.3f}%. Absorbed-sink fraction {sys_pct:.1f}% is a model output, "
          "not validated human bioavailability or serum exposure. LSODA solver used.")

cap(doc, "Table S2. Pharmacometric PK/PD indices from published aggregate GCF anchors. For chlorhexidine the index is "
         "time above a BIOCIDAL threshold (125 µg/mL), NOT an antibiotic MIC.")
add_df_table(doc, s2)

cap(doc, "Table S3. Three-dosage-form calibration to published summary anchors (each product fit with its "
         "formulation-appropriate release representation).")
add_df_table(doc, s3)
note(doc, "AAFE, absolute average fold error. AAFE summarizes calibration to sparse aggregate anchors only. No patient-level or independent external validation is claimed.")

cap(doc, "Table S4. Budget-impact analysis - two scenarios, both consistent with the Markov disease-cost stream.")
add_df_table(doc, s4)
note(doc, "The downstream saving is DERIVED from the Markov disease-cost stream, so BIA and CEA agree in sign "
          "(net cost-incurring). Break-even saving = each arm's course cost.")

cap(doc, "Table S5. Value of information (subgroup-weighted PSA; five-fold cross-fitted Strong-Oakley-Brennan GAM metamodels).")
add_df_table(doc, s5)
note(doc, f"Resolving closure/efficacy (the PPD weighted-mean-difference, dominated by the chlorhexidine effect) "
          f"accounts for ~{100*evppi['Closure/efficacy (PPD WMD)']/EVPI:.0f}% of the EVPI (${EVPI:,.0f}/site). "
          "EVPPI groups are not additive (each <= EVPI; shares need not sum to 100% because of interactions).")



cap(doc, "Table S5. Release-representation identifiability diagnostics from multistart calibration and the residual Jacobian.")
add_df_table(doc, s3b)
note(doc, "The full local PBPK parameter set is not identifiable from sparse aggregate GCF anchors. These diagnostics do not constitute external validation.")

cap(doc, "Table S6. Deterministic sensitivity to alternative PPD-to-pocket-closure mappings.")
add_df_table(doc, s6)
note(doc, "All mappings are structural scenarios calibrated to the same baseline and local slope. Monotone mappings preserve input-effect ordering and therefore cannot validate that ordering.")

cap(doc, "Table S7. Sensitivity to where the closure proxy enters the Markov model.")
add_df_table(doc, s6b)
note(doc, "The conservative base case uses closure only for the post-treatment initial state. Transition-only and dual-entry scenarios test structural coupling; dual entry reuses the same proxy and may amplify the PPD signal.")

cap(doc, "Table S8. Same-draw probabilistic sensitivity under alternative closure mappings.")
add_df_table(doc, s7)
note(doc, "Equal mapping weights are a transparent scenario mixture and are not posterior model probabilities. For the mixture, Monte Carlo SE uses each PSA draw as a cluster across the four correlated mappings.")

cap(doc, "Table S9. Sensitivity excluding MMP-8 QALYs and using mutually exclusive host-modulation pathways.")
add_df_table(doc, s8)
note(doc, "The evidence-audited base case sets the MMP-8 QALY increment to zero. Other rows are exploratory scenarios only.")

cap(doc, "Table S10. Provenance and limitations of economic and subgroup inputs.")
add_df_table(doc, s9)
note(doc, "These inputs are illustrative structural assumptions; resulting ICER, EVPI and optimal-strategy outputs are not decision-grade estimates.")


doc.save(os.path.join(TABDIR, "Tables.docx"))
_csvs = sorted(os.listdir(TABDIR))
print(f"Exported tables (manuscript index order) to {TABDIR}/:")
for f in _csvs: print("  -", f)

# Markdown cell 26
# ## Verified sources used for corrected parameters
#
# - Yamauchi S, Inoue D, Sugano K. *ADMET & DMPK*. 2020;8:129-138. DOI: 10.5599/admet.797. PMID: 35300369.
# - Stoller NH et al. *J Periodontol*. 1998;69:1085-1091. DOI: 10.1902/jop.1998.69.10.1085.
# - Soskolne WA et al. *J Clin Periodontol*. 1998;25:1017-1021. DOI: 10.1111/j.1600-051x.1998.tb02407.x. PMID: 9869352.
# - FDA ATRIDOX label / NDA 50-751: https://www.accessdata.fda.gov/drugsatfda_docs/label/2011/050751s015lbl.pdf
# - FDA PerioChip label / NDA 20-774: https://www.accessdata.fda.gov/drugsatfda_docs/label/2011/020774Orig1s010lbl.pdf
# - FDA ARESTIN clinical-pharmacology review / NDA 50-781: https://www.accessdata.fda.gov/drugsatfda_docs/nda/2001/50781_Arestin_biopharmr.pdf
# - Soysa NS, Jayakody SL, Alles CNRA. *Front Dent Med*. 2025;6:1658720. DOI: 10.3389/fdmed.2025.1658720. PMID: 41079915.
# - Ma L, Diao X. *BMC Oral Health*. 2020;20:262. DOI: 10.1186/s12903-020-01247-8. PMID: 32957945.
# - Annisa ZU et al. *BMC Oral Health*. 2023;23:819. DOI: 10.1186/s12903-023-03241-2. PMID: 37899443. Used only to document why 0.50-0.58 mm is **not** an SRP-controlled CHX effect range.
# - Burns T et al. *Antimicrob Agents Chemother*. 1992;36:227-230. DOI: 10.1128/AAC.36.1.227. PMID: 1317148 (doxycycline IC50 15-30 uM for neutrophil/GCF collagenases).
#
# ### Interpretation boundary
#
# The local PK anchors and meta-analytic PPD/CAL effects are literature-derived. The PPD-to-closure mapping, recurrence model, Markov transitions, utilities, costs, subgroup prevalences/modifiers, regimen objective, and all resulting ICER/EVPI/EVIC numbers are **illustrative structural outputs**, not empirical estimates. They require external calibration before scientific or policy claims are made.
